# Yu, Gutnisky, Hires & Svoboda (2016) on CRCNS ssc-7

Paper. Yu J, Gutnisky DA, Hires SA, Svoboda K (2016). Layer 4 fast-spiking interneurons filter
thalamocortical signals during active somatosensation. *Nature Neuroscience* 19, 1647-1657.
[doi:10.1038/nn.4412](https://doi.org/10.1038/nn.4412). The claims file was written from the
version of record (main text, figure legends, Online Methods) and its supplement.

Data set. CRCNS [ssc-7](https://doi.org/10.6080/K0T43R01) (doi:10.6080/K0T43R01), read anonymously
from the public bucket `crcnsarchive` at `ssc-7/live/`. CRCNS data sets are not versioned, so the
version is recorded as `unversioned`, the only version there is; `ssc-7/live/` is the published
copy (the `orig/` folder holds only a read-me). The files read and their MD5 sums from
`checksums.md5`: `data.tar.gz` 670a36de4916084ff3384d7e9450e6a8 (6,207,296,281 bytes, object last
modified 2026-02-11) and `docs/yu_gutnisky_CellList.xlsx` 9b0d4c9f4749de6d30fe6a17726f4754 (read
while preparing, for the example-cell names; the files' own unit descriptions are used here). The
data descriptor (version 0.6, July 27, 2017) and the example code `example_analysis_code.tar.gz`
(c178aa213fa11b4974023e147a26e637) were read while preparing; the code is cited where it settles a
detail.

Reading the data. `data.tar.gz` is a single gzipped tar of 148 NWB 1.0 files. A gzip stream
cannot be opened in place, so the notebook streams it once from the start (about twelve to fourteen minutes) and
reduces each file in memory as it passes (`ssc7.py`). Nothing is written to disk.

What the data set holds. Seven folders: 29 VPM units (18 silicon-probe sessions), 37
whole-cell and 48 cell-attached L4 excitatory neurons, 7 whole-cell and 19 cell-attached L4 FS
neurons, and after infraorbital nerve (ION) cut 9 whole-cell L4 excitatory and 10 cell-attached
L4 FS neurons. Not in it: the 17 thalamic multiunit recordings of Fig. 6b,c, the rabies tracing,
the L5 FS neurons after ION cut, and any recording with the depolarizing current injection of
Fig. 3 (no whole-cell L4 excitatory trial has a median Vm above -30 mV).

| Claim | Tested |
|---|---|
| behavior-performance | yes |
| vpm-baseline-rate | yes |
| vpm-touch-response | yes |
| vpm-whisking-count | yes |
| vpm-whisking-rate | yes |
| vpm-phase-modulation | yes |
| vpm-rate-amplitude-trials | yes |
| l4e-example-e1 | yes |
| l4e-example-e24-e32 | yes |
| l4e-resting-vm | yes |
| l4e-touch-psp | yes |
| l4e-whisking-vm | yes |
| l4e-distance-threshold | yes |
| l4e-reversal-potential | yes |
| l4e-hf-fluctuations | yes |
| l4e-inhibition-window | no, data missing |
| l4e-whisking-inhibition | no, data missing |
| fs-baseline-rate | yes |
| fs-touch-vm | yes |
| fs-touch-spikes | yes |
| fs-whisking-count | yes |
| fs-whisking-vm | yes |
| fs-whisking-rate | yes |
| fs-phase-coupling | yes |
| fs-hf-fluctuations | yes |
| l4e-spikes-touch-selective | yes |
| thalamus-inhibition-vpm | no, data missing |
| thalamus-inhibition-l4e-vm | yes |
| thalamus-inhibition-fs | yes |
| thalamus-inhibition-dose | yes |
| rabies-vpm-input | no, data missing |
| ion-cut-behavior | yes |
| ion-cut-l4e-touch | yes |
| ion-cut-l4e-whisking | yes |
| ion-cut-l4e-phase | yes |
| ion-cut-l4e-spectra | yes |
| ion-cut-fs-touch | yes |
| ion-cut-fs-whisking | yes |
| ion-cut-fs-phase | yes |
| ion-cut-l5-fs | no, data missing |
| enphr-kruskal-wallis | yes |
| enphr-l4e-rates | yes |
| enphr-whisking-driven | yes |
| enphr-spikes-per-touch | yes |
| enphr-auc | yes |
| enphr-auc-groups | yes |
| enphr-pv-latency | yes |
| enphr-behavior | yes |

The outcomes are given in the summary at the end and in the attempt record.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import ssc7

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="Unknown extension")  # openpyxl, reading the cell list
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()
cells = ssc7.load(verbose=False)
print(f"read {len(cells)} units from {ssc7.DATA_URL} in {time.time() - started:.0f} s; "
      f"files that could not be read: {ssc7.load.failed}")
NAME = {ssc7.cell_name(c): c for c in cells}


def ci(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return f"mean {x.mean():.3g} +- {x.std(ddof=1):.3g} s.d., median {np.median(x):.3g}, n = {len(x)}"


def within(v, e, t):
    return bool(np.isfinite(v) and abs(v - e) <= t + 1e-12)

## Data check

Counts by folder, mouse and recording, sampling rates, trial counts and the ranges of the
variables used. Each check of the description against the data is stated below the output.

In [ ]:
rows = []
for c in cells:
    m = c["meta"]
    rows.append(dict(group=ssc7.group(c), name=ssc7.cell_name(c), file=m["file"], subject=m["subject"],
                     unit=m.get("unit_description", ""), n_trials=m["n_trials"], good=int(c["good"].sum()),
                     stim=int(c["stim"].sum()), ephys_hz=round(m.get("ephys_fs", 0)),
                     n_whiskers=m["n_whiskers"], pw=str(m["principal_whisker"]),
                     config=str(m["whisker_configuration"]), pw_series=ssc7.pw_index(c),
                     notes=m["notes"].replace("Experiment type: behavior, ", "")))
info = pd.DataFrame(rows)
print("Units per folder group:")
print(info.group.value_counts().to_string())
print("\nFiles:", info.file.nunique(), "; mice (subject_id, cortex):", info[info.group != "VPM"].subject.nunique())
print("\nEphys sampling rates (Hz) in cortical files:", info[info.group != "VPM"].ephys_hz.value_counts().to_dict())
print("Trials per unit: median", info.n_trials.median(), "range", info.n_trials.min(), "-", info.n_trials.max())
print("\nUnit descriptions in folders that are not their own type:")
print(info[(info.group.str.startswith("FS")) & info.unit.str.contains("xcit")][["group", "name", "unit"]].to_string(index=False))
print("\nCortical files with more than one tracked whisker series:", int(((info.group != "VPM") & (info.n_whiskers > 1)).sum()),
      "; VPM sessions with more than one:", info[(info.group == "VPM") & (info.n_whiskers > 1)].file.nunique())
# recordings deposited twice: identical spike-count and trial-count signatures
sig = {}
for c in cells:
    st = np.atleast_1d(c["spike_times"])
    key = (c["meta"]["n_trials"], len(st), round(float(st[:5].sum()), 4) if len(st) else 0.0)
    sig.setdefault(key, []).append(ssc7.cell_name(c))
print("Units sharing trial count, spike count and first spike times (duplicates):", [v for v in sig.values() if len(v) > 1])
# clocks
d = []
for c in cells:
    if ssc7.group(c) == "VPM":
        continue
    d.append((round(c["meta"]["ephys_fs"]), np.nanmin(c["vm_t0"]), np.nanmax(c["vm_t0"])))
d = pd.DataFrame(d, columns=["hz", "min", "max"]).round(4)
print("\nEphys trial start minus whisker trial start (s), after converting the whisker trial start from ms:")
print(d.groupby("hz").agg(["min", "max"]).to_string())
spk_rel = np.concatenate([np.concatenate(ssc7.spikes_by_trial(c)) for c in cells])
print(f"\nSpike times relative to the trial's first whisker frame: 1st-99th percentile "
      f"{np.percentile(spk_rel, 1):.3f} to {np.percentile(spk_rel, 99):.3f} s (whisker video 5 s per trial)")
ang = np.concatenate([c["angle"][ssc7.pw_index(c)][np.isfinite(c["angle"][ssc7.pw_index(c)])].ravel()[::50] for c in cells])
print(f"Whisker angle (degrees, units not stated in the files): 1st-99th percentile {np.percentile(ang, 1):.1f} to {np.percentile(ang, 99):.1f}")
vmall = np.concatenate([np.concatenate([v[::200] for v in c["vm"]]) for c in cells if c["vm"]])
print(f"Whole-cell Vm (mV): 1st-99th percentile {np.percentile(vmall, 1):.1f} to {np.percentile(vmall, 99):.1f}")
# laser series and stimulus types
lz = []
for c in cells:
    if ssc7.group(c) == "VPM":
        continue
    n_on = 0
    if c["laser"]:
        n_on = int(sum(ssc7.laser_power(c, i) > 0.05 for i in range(c["meta"]["n_trials"])))
    kinds = pd.Series([x.replace("Go", "").replace("Nogo", "") or "none" for x in c["epoch_description"]]).value_counts().to_dict()
    lz.append(dict(group=ssc7.group(c), name=ssc7.cell_name(c), stim_trials=int(c["stim"].sum()), trials_with_laser_trace=n_on, **{f"type{k}": v for k, v in kinds.items()}))
lz = pd.DataFrame(lz).fillna(0)
print("\nStimulus trials and trials whose stored photostimulus trace is above 0.05 mW, summed by group:")
print(lz.groupby("group").sum(numeric_only=True).astype(int).to_string())
print("Most trials with a stimulus flag in a file:", lz.stim_trials.max(), "; most trials with a nonzero laser trace in a file:", lz.trials_with_laser_trace.max())
print("\nCell list check: whole-cell L4E 37, cell-attached L4E 48, FS 7 + 19, ION cut 9 + 10, VPM 29 units in 18 sessions:",
      info.groupby("group").size().to_dict(), "; VPM sessions:", info[info.group == "VPM"].file.nunique())
print("\nTrials tagged passive touch:", int(sum(sum("passive" in str(t) for t in c["epoch_tag"]) for c in cells)))

# Trial-type columns: the names each folder gives to the columns of analysis/trial_type_mat (the
# performance and behavior claims take columns 1 to 4 as Hit, Miss, CR, FA)
print("\nTrial-type column names by folder:")
for g in sorted(info.group.unique()):
    nm = {tuple(c["meta"]["trial_type_names"][:6]) for c in cells if ssc7.group(c) == g}
    print(f"  {g}: {nm}")
# Pole times relative to each trial's first whisker frame (s), used by M-ROC reading (a)
pr = []
for c in cells:
    if c["pole_in"] is None or ssc7.group(c) == "VPM":
        continue
    n_ = min(len(c["t0"]), len(np.atleast_1d(c["pole_in"])), len(np.atleast_1d(c["pole_out"])))
    pr.append((ssc7.group(c), np.nanmin(np.asarray(c["pole_in"])[:n_] - c["t0"][:n_]), np.nanmax(np.asarray(c["pole_in"])[:n_] - c["t0"][:n_]),
               np.nanmin(np.asarray(c["pole_out"])[:n_] - c["t0"][:n_]), np.nanmax(np.asarray(c["pole_out"])[:n_] - c["t0"][:n_])))
pr = pd.DataFrame(pr, columns=["group", "pole_in_min", "pole_in_max", "pole_out_min", "pole_out_max"])
print("\nPole-in and pole-out times minus the trial's first whisker frame (s), cortical files:")
print(pr.groupby("group").agg(pole_in_min=("pole_in_min", "min"), pole_in_max=("pole_in_max", "max"),
                              pole_out_min=("pole_out_min", "min"), pole_out_max=("pole_out_max", "max")).round(2).to_string())
# Whole-cell recordings: every ephys trial against the whisker trials, the per-trial median Vm and
# the largest injected current, to find depolarized trials (Fig. 3) and drifting recordings
wc_rows = []
for c in cells:
    if c["vm"] is None:
        continue
    e = c["ephys_trials"]
    med = e["median_vm"]
    st_ = ssc7.stable_trials(c)
    wc_rows.append(dict(group=ssc7.group(c), name=ssc7.cell_name(c), n_trials=c["meta"]["n_trials"], n_ephys_trials=c["meta"]["n_ephys_trials"],
                        median_vm=round(float(np.nanmedian(med)), 1), max_trial_median_vm=round(float(np.nanmax(med)), 1),
                        trials_above_minus30=int((med > -30).sum()), max_abs_current_nA=round(float(np.nanmax(e["max_abs_current"])), 2),
                        trials_far_from_median=int((~st_[: len(c["vm"])]).sum())))
WCT = pd.DataFrame(wc_rows)
print("\nWhole-cell neurons whose ephys trial count differs from the whisker trial count:", WCT[WCT.n_trials != WCT.n_ephys_trials].name.tolist())
print("Largest per-trial median Vm (mV) and largest |current| (nA) by group, over every ephys trial:")
print(WCT.groupby("group").agg(neurons=("name", "size"), max_trial_median_vm=("max_trial_median_vm", "max"), max_abs_current_nA=("max_abs_current_nA", "max"),
                               trials_above_minus30=("trials_above_minus30", "sum"), trials_far_from_median=("trials_far_from_median", "sum")).to_string())
print("Neurons with a trial above -30 mV, a current above 0.5 nA, or a trial more than 20 mV from their median:")
print(WCT[(WCT.trials_above_minus30 > 0) | (WCT.max_abs_current_nA > 0.5) | (WCT.trials_far_from_median > 0)].to_string(index=False))
# The data set's cell list (docs/yu_gutnisky_CellList.xlsx), against the folders
CL = ssc7.read_cell_list()
key = lambda s: str(s).replace("normal", "").replace("_", "-")[:24]
cl_keys = {key(x): sec for x, sec in zip(CL.cell, CL.section)}
info["cell_list_section"] = [cl_keys.get(key(f.replace(".nwb", "")), cl_keys.get(key(f.replace(".nwb", "")[:6]), "")) for f in info.file]
print("\nCell list sections against the data set's folders (cortical units):")
print(pd.crosstab(info[info.group != "VPM"].group, info[info.group != "VPM"].cell_list_section).to_string())
print("Cortical units not found in the cell list:", info[(info.group != "VPM") & (info.cell_list_section == "")].file.tolist())

### Checks of the description against the data

- Counts. The folders hold the counts the claims file gives from the paper's Supplementary Table 2:
  37 whole-cell and 48 cell-attached L4 E neurons, 26 FS neurons (7 + 19), 9 + 10 after ION cut,
  29 VPM units in 18 sessions. No two units share trial count, spike count and first spike
  times, so no recording is deposited twice. Seven VPM sessions come from one mouse (ANM199552)
  with different unit names each day; they are treated as different units, as the paper's count
  of 29 does.
- Unit labels. One neuron in the whole-cell FS folder, JY1139, is described in its file as
  "Excitatory"; the folder and the cell list (section "Whole-cell, FS neurons", a PV-ChR2 mouse)
  call it FS, and it fires 2.7 spikes per touch, as FS neurons do. The folder is used.
- Cell list. The data set's cell list (`docs/yu_gutnisky_CellList.xlsx`, sheet
  AllCellsReported) puts every cortical unit in the section of its folder, except JY1268 (in the
  cell-attached L4 E folder), which the cell list does not name; its "Extracellular units"
  section has 47 rows against 48 files.
- Trial types. Every folder names the columns of `trial_type_mat` Hit, Miss, CR, FA (and
  Stimtrials in the cortical folders), the order the behavior claims use.
- Whole-cell trials. Every whole-cell file has as many ephys trials as whisker trials, so no
  ephys trial is dropped. No L4 E trial has a median Vm above -42.8 mV, and no L4 E current
  exceeds 0.2 nA: the depolarized recordings of Fig. 3 are not here, including those of JY0978,
  which the cell list names as the Fig. 3 example. One FS neuron, JY1139, starts near 0 mV
  (trials 0 to 17 above -30 mV) and settles at about -41 mV from trial 20, while it keeps firing
  (8 to 85 spikes per trial) with no current above 0.1 nA. That is a drifting recording, not the
  depolarization protocol, which inactivates spikes with about 1 nA. Trials whose median Vm lies
  more than 20 mV from the neuron's median are left out of the Vm measures (13 trials, all of
  JY1139; 5 of them control trials), after the Methods' rejection of recordings "with slow
  baseline drift (>20 mV)" (an assumption; the FS Vm claims show both).
- Pole times. Pole-in comes 0.83 s after the first whisker frame in nearly every cortical trial;
  pole-out times run from 2.4 s to well past the 5 s of video, so M-ROC reading (a) bounds its
  reference times by the video.
- Clocks. In the cortical files the whisker timestamps are the trial start in milliseconds plus the
  time within the trial in seconds (frame spacing 0.001 s, trial starts 1,000 times the ephys trial
  starts). Once the trial start is read as milliseconds, every ephys trial starts 10 ms before the
  first whisker frame (10 kHz files) or with it (20 kHz files). Spikes (ephys clock) aligned to touch
  onsets (whisker clock) peak 2 ms (VPM) and 5 ms (FS) after touch, and the peak vanishes when the
  touch times are shifted by 0.5 to 1.5 s, so the two clocks are consistent with each other, to
  within the 1 ms of the whisker frames. The example neuron E No. 1 below gives the paper's touch
  latency (4.8 against 4.7 ms). That is consistent with a shared clock but does not prove it.
- Units. Vm is in mV, laser power in mW, whisker angle in degrees (inferred from the range, -37 to
  43 degrees; the files say "unknown"). The ephys sampling rate is 10 or 20 kHz, as the Methods
  say; 20 kHz Vm is averaged in pairs to 10 kHz.
- Principal whisker. Seventy cortical files hold two tracked whisker series, and the files do not
  say which is which. Mapping the series to `whisker_configuration` in order gives FS neurons with
  almost no touch response (0.45 spikes per touch against 1.78 with series 1; table above). The
  data set's example code always takes the first whisker as the main one (`mainwid = 0` in
  `ConvertfromNWB.m`). Series 1 is used as the principal whisker (an assumption). This is recorded
  as a finding.
- Light trials. Each file stores one photostimulus series per light site, sampled with the ephys.
  The series give each trial's sites, power (3 mW at 594 nm; 3.8, 7.5 or 15 mW at 473 nm) and
  light period, and the light sites agree with the experiments of the Online Methods. In most
  files the light is on exactly in the trials the `Stimtrials` flag marks (the table above). Eight
  whole-cell L4 E files flag 132 stimulus trials but store no photostimulus series, and in two
  more (Exp_2015-09-24_004, JY1633) 23 flagged trials have no light in their series. Those
  trials count as neither control nor light trials.
- 410 trials are tagged "passive touch"; their first touch is left out, as the example code does.

In [ ]:
# Which tracked whisker series is the principal whisker: touch responses of the FS neurons (the
# strongest touch responders) with the series each reading picks, in files with two series.
rows = []
for c in cells:
    if ssc7.group(c) not in ("FS_wc", "FS_ca", "L4E_wc", "L4E_ca") or c["meta"]["n_whiskers"] < 2:
        continue
    spk = ssc7.spikes_by_trial(c)
    tr = ssc7.control_trials(c)
    r = dict(group=ssc7.group(c), name=ssc7.cell_name(c), config=str(c["meta"]["whisker_configuration"]), pw=str(c["meta"]["principal_whisker"]))
    for rd in ("code", "config"):
        ssc7.PW_READING = rd
        tt = ssc7.touch_table(c, 0.025, tr)
        r[f"series_{rd}"] = ssc7.pw_index(c)
        r[f"spt_{rd}"] = ssc7.spikes_per_touch(spk, tt) if len(tt) else np.nan
    ssc7.PW_READING = "code"
    rows.append(r)
PWT = pd.DataFrame(rows)
print("files with two tracked whiskers:", len(PWT), "; the readings pick different series in", int((PWT.series_code != PWT.series_config).sum()))
fs2 = PWT[PWT.group.str.startswith("FS")]
print(fs2.round(2).to_string(index=False))
print(f"FS neurons with two series: mean spikes per touch, reading 'code' (series 1) {fs2.spt_code.mean():.2f}, reading 'config' {fs2.spt_config.mean():.2f}")
e2 = PWT[PWT.group.str.startswith("L4E")]
print(f"L4 E neurons with two series: {len(e2)}; mean spikes per touch, reading 'code' {e2.spt_code.mean():.3f}, reading 'config' {e2.spt_config.mean():.3f}; "
      f"the readings pick different series in {int((e2.series_code != e2.series_config).sum())}")
print(e2.round(3).to_string(index=False))

### Per-unit measures used by the claims

Spike measures for every unit, in control trials (good trials without a stimulus flag), under
both readings of the nonwhisking margin (M-EPOCH (a) and (b)); Vm measures for every
whole-cell neuron. Each claim below uses these.

In [ ]:
t1 = time.time()
S = {"a": {}, "b": {}}
for c in cells:
    for mg in "ab":
        S[mg][ssc7.cell_name(c)] = ssc7.spike_summary(c, margin=mg)
V = {}
for c in cells:
    if c["vm"] is not None:
        V[ssc7.cell_name(c)] = ssc7.vm_summary(c)
print(f"spike summaries for {len(S['a'])} units, Vm summaries for {len(V)} neurons in {time.time() - t1:.0f} s")
GROUP = {ssc7.cell_name(c): ssc7.group(c) for c in cells}
names = lambda *g: [n for n in S["a"] if GROUP[n] in g]
VPM, FS, L4E85 = names("VPM"), names("FS_wc", "FS_ca"), names("L4E_wc", "L4E_ca")
L4EWC, FSWC, L4EION, FSION = names("L4E_wc"), names("FS_wc"), names("L4E_ion"), names("FS_ion")
print({k: len(v) for k, v in dict(VPM=VPM, FS=FS, L4E85=L4E85, L4EWC=L4EWC, FSWC=FSWC, L4EION=L4EION, FSION=FSION).items()})
drop = pd.DataFrame([dict(name=n, group=GROUP[n], control_trials=S["a"][n]["n_trials"], touches=S["a"][n]["n_touch"],
                          whisk_s=round(S["a"][n]["whisk_s"], 1), nonw_s=round(S["a"][n]["nonw_s"], 1)) for n in S["a"]])
print("\nUnits with no nonwhisking or no whisking epoch (reading a):",
      drop[(drop.whisk_s == 0) | (drop.nonw_s == 0)][["name", "group", "control_trials", "whisk_s", "nonw_s"]].to_string(index=False))
print("Units with no touch (ITI >= 25 ms):", drop[drop.touches == 0][["name", "group"]].to_string(index=False))
print("\nEpoch time per unit (s), reading a: whisking median", drop.whisk_s.median(), "; nonwhisking median", drop.nonw_s.median())
# Clock check: spikes (ephys clock) aligned to touch onsets (whisker clock). A shared clock gives a
# peak a few ms after touch; with touch times shifted by a random 0.5-1.5 s the peak must vanish.
rng0 = np.random.default_rng(0)
for g, nn in (("VPM", VPM), ("FS", FS), ("L4E", L4E85)):
    ps = np.sum([S["a"][n]["psth"] for n in nn], 0) / sum(S["a"][n]["n_touch"] for n in nn) * 1000
    sh = []
    for n in nn:
        tt = S["a"][n]["tt"].copy()
        if len(tt):
            tt[:, 1] = np.clip(tt[:, 1] + rng0.uniform(0.5, 1.5, len(tt)) * rng0.choice([-1, 1], len(tt)), 0.2, 4.8)
            sh.append(ssc7.touch_psth(S["a"][n]["spk"], tt).sum(0))
    sh = np.sum(sh, 0) / sum(S["a"][n]["n_touch"] for n in nn) * 1000
    print(f"{g}: touch PSTH peak {ps[100:150].max():.0f} spikes/s at {np.argmax(ps[100:150])} ms after onset, baseline (-25-0 ms) {ps[75:100].mean():.0f}; "
          f"with shifted touch times peak {sh[100:150].max():.0f} at {np.argmax(sh[100:150])} ms, baseline {sh[75:100].mean():.0f}")


def msig(s, reading):
    """M-SIG: sign of a significant whisking effect, two-sided rank-sum at p < 0.05."""
    if reading == "a":
        w, n = s["w_rows"], s["n_rows"]
        if len(w) < 2 or len(n) < 2:
            return 0, np.nan
        x, y = w[:, 1] / w[:, 2], n[:, 1] / n[:, 2]
    else:
        x, y = s["w_trial"], s["n_trial"]
        if len(x) < 2 or len(y) < 2:
            return 0, np.nan
    p = stats.ranksums(x, y).pvalue
    if not np.isfinite(p) or p >= 0.05:
        return 0, p
    return (1 if np.mean(x) > np.mean(y) else -1), p

# Variants used by several claims below:
# - the principal whisker taken from the whisker configuration (ssc7.pw_index, reading "config"),
#   for the L4 E neurons (spike and Vm measures);
# - FS whole-cell Vm measures with every control trial, without leaving out trials far from the
#   neuron's usual Vm (ssc7.stable_trials).
t1 = time.time()
ssc7.PW_READING = "config"
S_cfg = {n: ssc7.spike_summary(NAME[n], margin="a") for n in L4E85}
V_cfg = {n: ssc7.vm_summary(NAME[n]) for n in L4EWC}
ssc7.PW_READING = "code"
V_all = {n: ssc7.vm_summary(NAME[n], drop_drift=False) for n in FSWC}
n_diff = int((PWT.group.str.startswith("L4E") & (PWT.series_code != PWT.series_config)).sum())
print(f"variant summaries in {time.time() - t1:.0f} s; L4 E neurons whose principal series differs between the readings: {n_diff}")
print("trials left out as far from the neuron's usual Vm, per FS whole-cell neuron:", {n: int(V_all[n]["n_trials"] - V[n]["n_trials"]) for n in FSWC})

## behavior-performance

> Mice performed the task at 0.72 +- 0.11 correct during the cortical recordings (111 cells)
> and 0.71 +- 0.07 during the VPM recordings (29 cells); they touched the pole 5.7 +- 5.3 times
> per trial at a proximal location (4,147 trials) and 0.9 +- 3.4 times at a distal location
> (4,173 trials), and licked 286 +- 329 ms after the first touch.

Correct = Hit or CR in the file's trial-type matrix; proximal = Go (lick) trials, distal = Nogo
trials (Online Methods, Behavior: mice lick for the posterior positions). Cortical cells are
the 111 non-ION-cut cortical recordings; touch counts pool the cortical and VPM recordings
(each VPM session once). Readings: trials (a) all good trials, (b) control trials;
touches counted (c) over the whole trial, (d) before the first lick.

In [ ]:
def behav(c, trials):
    tt = c["trial_type"]
    typed = trials & (tt[:, :4].sum(1) > 0)
    corr = (tt[:, 0] + tt[:, 2]) > 0
    go = (tt[:, 0] + tt[:, 1]) > 0
    T = ssc7.touches(c)
    L = ssc7.licks_by_trial(c)
    rows = []
    for i in np.flatnonzero(typed):
        a = T[i][0]
        lk = L[i][(L[i] >= 0) & (L[i] < 5)]
        fl = lk[0] if len(lk) else np.inf
        rt = np.nan
        if len(a):
            after = lk[lk > a[0]]
            rt = (after[0] - a[0]) * 1000 if len(after) else np.nan
        rows.append((go[i], corr[i], len(a), int(np.sum(a < fl)), rt))
    return np.array(rows, float).reshape(-1, 5)


cort = [c for c in cells if ssc7.group(c) in ("L4E_wc", "L4E_ca", "FS_wc", "FS_ca")]
vpm_sessions = {}
for c in cells:
    if ssc7.group(c) == "VPM":
        vpm_sessions.setdefault(c["meta"]["file"], []).append(c)
print("cortical cells:", len(cort), "; VPM units:", sum(len(v) for v in vpm_sessions.values()), "in", len(vpm_sessions), "sessions")
res_beh = {}
for rd in "ab":
    sel = (lambda c: c["good"]) if rd == "a" else ssc7.control_trials
    pc_cort = [behav(c, sel(c))[:, 1].mean() for c in cort]
    # VPM: one performance per unit (the 29 cells), from the session's good trials of that unit
    pc_vpm = [behav(c, sel(c))[:, 1].mean() for v in vpm_sessions.values() for c in v]
    allrows = np.vstack([behav(c, sel(c)) for c in cort] + [behav(v[0], sel(v[0])) for v in vpm_sessions.values()])
    go, nogo = allrows[allrows[:, 0] == 1], allrows[allrows[:, 0] == 0]
    rt = allrows[:, 4][np.isfinite(allrows[:, 4])]
    for cnt, col in (("c", 2), ("d", 3)):
        vals = [np.mean(pc_cort), np.mean(pc_vpm), go[:, col].mean(), nogo[:, col].mean(), rt.mean()]
        res_beh[rd + cnt] = vals
        print(f"reading ({rd},{cnt}): correct cortical {np.mean(pc_cort):.3f} +- {np.std(pc_cort, ddof=1):.3f} (n={len(pc_cort)}); "
              f"VPM {np.mean(pc_vpm):.3f} +- {np.std(pc_vpm, ddof=1):.3f} (n={len(pc_vpm)}); touches Go {go[:, col].mean():.2f} +- {go[:, col].std():.2f} "
              f"({len(go)} trials), Nogo {nogo[:, col].mean():.2f} +- {nogo[:, col].std():.2f} ({len(nogo)} trials); "
              f"reaction time {rt.mean():.0f} +- {rt.std():.0f} ms (n={len(rt)})")
exp_beh = [(0.72, 0.072), (0.71, 0.071), (5.7, 0.57), (0.9, 0.09), (286, 28.6)]
for k, v in res_beh.items():
    print(k, ["%s" % within(x, e, t) for x, (e, t) in zip(v, exp_beh)])
# variant: touches counted on the series the whisker configuration names as principal
ssc7.PW_READING = "config"
allv = np.vstack([behav(c, c["good"]) for c in cort] + [behav(v[0], v[0]["good"]) for v in vpm_sessions.values()])
ssc7.PW_READING = "code"
print("variant, principal whisker by the configuration (reading a,c): Go touches %.2f, Nogo %.2f; reaction time %.0f ms" % (
    allv[allv[:, 0] == 1, 2].mean(), allv[allv[:, 0] == 0, 2].mean(), np.nanmean(allv[:, 4])))
# cortical touch counts only, as a variant
allc = np.vstack([behav(c, c["good"]) for c in cort])
print("variant, cortical recordings only (reading a,c): Go touches %.2f (%d trials), Nogo %.2f (%d trials)" % (
    allc[allc[:, 0] == 1, 2].mean(), (allc[:, 0] == 1).sum(), allc[allc[:, 0] == 0, 2].mean(), (allc[:, 0] == 0).sum()))

**Result: partially reproduced.** Cortical performance is 0.706 (all good trials) or 0.721 (control trials) correct, within 0.72 +- 0.072. VPM performance is 0.630, outside 0.71 +- 0.071. Touches per trial are 7.56 at the proximal (Go) pole and 1.06 at the distal pole over the whole trial, and 2.74 and 0.36 before the first lick. None of these is within 5.7 +- 0.57 and 0.9 +- 0.09. The reaction time is 328 to 340 ms, outside 286 +- 28.6. So one of the five values matches under every combination of readings. Proximal touches outnumber distal ones in every reading. With the principal whisker taken from the whisker configuration (variant), the Go count is 5.93 and the Nogo count 0.63.

## vpm-baseline-rate

> VPM neurons fire at substantial rates in the absence of whisker movement and touch (5.1 +- 5.5
> spikes/s, median 3.3, n = 29).

In [ ]:
for mg in "ab":
    for rr in "ab":
        x = [S[mg][n][f"nonw_{rr}"] for n in VPM]
        print(f"margin ({mg}), rate reading ({rr}): {ci(x)};  mean within 5.1+-1.02: {within(np.nanmean(x), 5.1, 1.02)}, median within 3.3+-0.33: {within(np.nanmedian(x), 3.3, 0.33)}")
x = np.array([S["a"][n]["nonw_a"] for n in VPM])
print("range", x.min().round(2), "to", x.max().round(2))
print("by mouse (reading a,a):", pd.Series(x, index=[NAME[n]["meta"]["file"][:9] for n in VPM]).groupby(level=0).agg(["mean", "count"]).round(2).to_dict("index"))
boot = np.random.default_rng(1)
b = [np.mean(boot.choice(x, len(x))) for _ in range(1000)]
print("bootstrap 95% CI of the population mean (1,000 iterations):", np.percentile(b, [2.5, 97.5]).round(2))

**Result: partially reproduced.** The mean nonwhisking rate of the 29 VPM units is 4.34 to 4.82 spikes/s under the four readings, within 5.1 +- 1.02. The median is 1.99 to 2.18 spikes/s, below 3.3 +- 0.33 under every reading. Units fire at baseline (range 0.03 to 17.66 spikes/s), but the distribution is more skewed than the paper's.

## vpm-touch-response

> VPM neurons respond to active touch with a sharp, short-latency, brief increase in spike rate
> (0.6 +- 0.5 spikes per touch, median 0.4; latency 3.2 +- 1.0 ms, median 2.5 ms; duration 2.9 +-
> 1.7 ms).

In [ ]:
spt = np.array([S["a"][n]["spt"] for n in VPM])
lat = np.array([S["a"][n]["latency"] for n in VPM])
dur = np.array([S["a"][n]["duration"] for n in VPM])
ntouch = np.array([S["a"][n]["n_touch"] for n in VPM])
print("touches per unit: median", np.median(ntouch), "range", ntouch.min(), "-", ntouch.max())
print("spikes per touch:", ci(spt), "range", np.nanmin(spt).round(2), "to", np.nanmax(spt).round(2))
print("latency (units passing the M-LAT exclusion):", ci(lat), "; excluded:", int(np.isnan(lat).sum()))
print("duration (25% of rate increase):", ci(dur[np.isfinite(lat)]), "(units with a latency)")
vals = [np.nanmean(spt), np.nanmedian(spt), np.nanmean(lat), np.nanmedian(lat), np.nanmean(dur[np.isfinite(lat)])]
exp = [(0.6, 0.093), (0.4, 0.04), (3.2, 0.32), (2.5, 0.25), (2.9, 0.32)]
print("within tolerance:", [within(v, e, t) for v, (e, t) in zip(vals, exp)])
pop = np.sum([S["a"][n]["psth"] for n in VPM], 0) / ntouch.sum() * 1000
print("population PSTH (spikes/s) at -3..+8 ms bins:", pop[97:109].round(1))
lat_c = np.array([ssc7.spike_latency(S["a"][n]["psth"], max(1, S["a"][n]["n_touch"]), centre=True) for n in VPM])
print(f"variant, latency at the centre of the crossing bin: {ci(lat_c)}; within (3.2+-0.32, 2.5+-0.25): {within(np.nanmean(lat_c), 3.2, 0.32)}, {within(np.nanmedian(lat_c), 2.5, 0.25)}")
print("latencies per unit (ms, left edge):", sorted(lat[np.isfinite(lat)].tolist()))

**Result: partially reproduced.** Spikes per touch are 0.50 +- 0.43 (median 0.363). The median is within 0.4 +- 0.04; the mean is just below 0.6 +- 0.093. The population PSTH peaks 2 ms after touch (139 spikes/s against a baseline of about 18). By M-LAT the latency is 2.48 ms on average (median 1 ms, n = 25), against 3.2 +- 0.32 and 2.5 +- 0.25; 13 of the 25 units cross in the first 1 ms bin. Read at the bin centre, the latency is 2.98 ms (within) with median 1.5 ms (outside). The duration is 4.08 ms against 2.9 +- 0.32. The response is sharp and brief, but here it is earlier and longer than the paper's. The 1 ms difference in the median latency is not explained.

## vpm-whisking-count

> Most VPM neurons (25 of 29) significantly increase their spike rate during whisking without
> touch; 3 of 29 do not increase but still respond to touch, and one decreases significantly.

In [ ]:
for mg in "ab":
    for rd in "ab":
        sg = [msig(S[mg][n], rd)[0] for n in VPM]
        up, dn, ns = sg.count(1), sg.count(-1), sg.count(0)
        print(f"margin ({mg}), samples ({rd}): increase {up}, decrease {dn}, no significant change {ns}; "
              f"within (25+-2.5, 3+-1, 1+-1): {within(up, 25, 2.5) and within(ns, 3, 1) and within(dn, 1, 1)}")
sg = [msig(S["a"][n], "a") for n in VPM]
print("p values, reading (a,a):", [f"{n}:{s}:{p:.2g}" for n, (s, p) in zip(VPM, sg)])

**Result: reproduced.** With per-trial rates (M-SIG reading b), 24 of 29 units increase significantly with whisking, 1 decreases and 4 show no significant change. That is within 25 +- 2.5, 3 +- 1 and 1 +- 1 under both nonwhisking margins. With per-epoch rates (reading a) the counts are 22 or 21 increases, 1 decrease and 6 or 7 unchanged, outside tolerance. The criterion holds under one reading. Whether the unchanged units respond to touch was not tested.

## vpm-whisking-rate

> The average VPM spike rate increases severalfold with whisker movement, from 5.1 +- 5.5 to
> 13.5 +- 12.7 spikes/s (n = 29, P = 4e-6, Wilcoxon signed-rank).

In [ ]:
for mg in "ab":
    for rr in "ab":
        nw = np.array([S[mg][n][f"nonw_{rr}"] for n in VPM])
        wh = np.array([S[mg][n][f"whisk_{rr}"] for n in VPM])
        p = stats.wilcoxon(wh, nw).pvalue
        print(f"reading (a) epochs, margin ({mg}), rate ({rr}): nonwhisking {nw.mean():.2f} +- {nw.std(ddof=1):.2f}, whisking {wh.mean():.2f} +- {wh.std(ddof=1):.2f}, "
              f"ratio {wh.mean() / nw.mean():.2f}, signed-rank p = {p:.2g}; within: {within(nw.mean(), 5.1, 1.02)}, {within(wh.mean(), 13.5, 2.36)}")
# variant (b): rate 0-400 ms after whisking onsets against 200 ms before
pre, post = [], []
for n in VPM:
    s = S["a"][n]
    a_, b_ = [], []
    for i, d in s["ep"].items():
        for e in d["onsets"]:
            t = e / 1000
            sp = s["spk"][i]
            a_.append(np.sum((sp >= t - 0.2) & (sp < t)) / 0.2)
            b_.append(np.sum((sp >= t) & (sp < t + 0.4)) / 0.4)
    pre.append(np.mean(a_) if a_ else np.nan)
    post.append(np.mean(b_) if b_ else np.nan)
pre, post = np.array(pre), np.array(post)
ok = np.isfinite(pre) & np.isfinite(post)
print(f"variant (b), onsets: before {np.nanmean(pre):.2f}, after {np.nanmean(post):.2f} spikes/s, n = {ok.sum()}, p = {stats.wilcoxon(post[ok], pre[ok]).pvalue:.2g}")

**Result: reproduced.** Under reading (a), whisking against nonwhisking epochs, the mean rate rises from 4.34 to 11.92 spikes/s (margin a, rate a; 2.74-fold), signed-rank p = 5.2e-8, n = 29. Both means are within tolerance (5.1 +- 1.02, 13.5 +- 2.36) under all four sub-readings. The onset variant (b) gives 4.61 to 9.92 spikes/s, p = 3.7e-9.

## vpm-phase-modulation

> VPM spike rate is modulated by the phase of whisker movement.

In [ ]:
for nb in (8, 16):
    r2 = np.array([S["a"][n][f"phase{nb}"][0] for n in VPM])
    cs = np.array([S["a"][n][f"phase{nb}"][1] for n in VPM])
    pp = np.array([S["a"][n][f"phase{nb}"][2] for n in VPM])
    good = r2 > 0.25
    print(f"{nb} bins: r2 > 0.25 in {good.sum()} of {len(VPM)}; r2 median {np.nanmedian(r2):.2f}; coupling strength of those, median {np.median(cs[good]):.2f} spikes/s "
          f"(range {cs[good].min():.2f}-{cs[good].max():.2f}); preferred phase median {np.median(pp[good]):.2f} rad")
# null: the same fit with spike counts shuffled across whisking samples, one unit per draw
rng = np.random.default_rng(2)
null = []
for n in VPM:
    s = S["a"][n]
    ph, y = ssc7.spike_phase_data(s["spk"], s["ep"])
    null.append(ssc7.phase_fit(ph, rng.permutation(y), 8)[0])
print("null (shuffled spikes), 8 bins: r2 > 0.25 in", int(np.sum(np.array(null) > 0.25)), "of", len(VPM))

**Result: reproduced.** The single-sinusoid fit has r2 > 0.25 in 27 of 29 units with 8 phase bins and 25 of 29 with 16 bins, more than half under both. The coupling strength of those units has a median of 1.63 spikes/s (range 0.13 to 6.54), and the preferred phase a median of -0.38 rad. The r2 cutoff is weak: with spike counts shuffled across whisking samples, 14 of 29 units still pass it.

## vpm-rate-amplitude-trials

> Within a single behavioral session, VPM spike rate and whisking amplitude co-vary across trials
> (Pearson r = 0.61, P = 1e-78, 310 trials).

In [ ]:
rows = []
for n in VPM:
    c = NAME[n]
    s = S["a"][n]
    rr, aa = [], []
    for i, d in s["ep"].items():
        rr.append(len(s["spk"][i][(s["spk"][i] >= 0) & (s["spk"][i] < 5)]) / 5.0)
        aa.append(np.nanmean(d["amp"]))
    r, p = stats.pearsonr(rr, aa)
    rows.append(dict(unit=n, session=c["meta"]["file"], trials=len(rr), r=round(r, 3), p=p))
ra = pd.DataFrame(rows)
print(ra.to_string(index=False))
print("units with 310 trials:", (ra.trials == 310).sum(), "; significant positive r:", int(((ra.r > 0) & (ra.p < 0.05)).sum()), "of", len(ra))
print("units with r within 0.61 +- 0.061 and p < 0.05:", ra[(abs(ra.r - 0.61) <= 0.061) & (ra.p < 0.05)].unit.tolist())

**Result: reproduced.** No unit has a session of 310 good trials, so the criterion's fallback applies. 24 of 29 units show a significant positive Pearson correlation between trial spike rate and mean whisking amplitude. Three units (3.3.33, 15.3.304, 11.2.110) have r within 0.61 +- 0.061.

## fs-baseline-rate

> L4 FS interneurons are active at baseline, even without whisker movement (9.3 +- 8.5 spikes/s,
> median 6.4, n = 26).

The neurons are the 26 in the data set's two control FS folders (7 whole-cell, 19
cell-attached).

In [ ]:
for mg in "ab":
    for rr in "ab":
        x = [S[mg][n][f"nonw_{rr}"] for n in FS]
        print(f"margin ({mg}), rate ({rr}): {ci(x)}; mean within 9.3+-1.67: {within(np.nanmean(x), 9.3, 1.67)}, median within 6.4+-0.64: {within(np.nanmedian(x), 6.4, 0.64)}")
x = np.array([S["a"][n]["nonw_a"] for n in FS])
print("range", x.min().round(2), "to", x.max().round(2))
print("whole-cell", ci(x[:0] if False else [S['a'][n]['nonw_a'] for n in FSWC]), "; cell-attached", ci([S['a'][n]['nonw_a'] for n in names('FS_ca')]))

**Result: reproduced.** The 26 FS neurons fire 9.43 spikes/s on average in nonwhisking epochs (median 6.3), within 9.3 +- 1.67 and 6.4 +- 0.64. All four readings fall within (means 9.43 to 9.96, medians 6.28 to 6.96).

## fs-touch-spikes

> L4 FS interneurons fire about two spikes per touch (1.9 +- 1.0, median 1.9, n = 26) with short
> latency (5.3 +- 2.1 ms, median 4.5 ms, n = 24).

In [ ]:
spt = np.array([S["a"][n]["spt"] for n in FS])
lat = np.array([S["a"][n]["latency"] for n in FS])
print("spikes per touch (0-50 ms minus 2 x 0-25 ms before):", ci(spt), "range", spt.min().round(2), "to", spt.max().round(2))
print("latency:", ci(lat), "; passing the exclusion:", int(np.isfinite(lat).sum()))
vals = [spt.mean(), np.median(spt), np.nanmean(lat), np.nanmedian(lat), np.isfinite(lat).sum()]
exp = [(1.9, 0.2), (1.9, 0.19), (5.3, 0.53), (4.5, 0.45), (24, 2.4)]
print("within tolerance:", [within(v, e, t) for v, (e, t) in zip(vals, exp)])
print("whole-cell", ci([S['a'][n]['spt'] for n in FSWC]), "; cell-attached", ci([S['a'][n]['spt'] for n in names('FS_ca')]))
# variant: all trials, not only control trials
spt_all = [ssc7.spike_summary(NAME[n], trials=NAME[n]["good"])["spt"] for n in FS]
print("variant, all good trials:", ci(spt_all))
lat_c = np.array([ssc7.spike_latency(S["a"][n]["psth"], max(1, S["a"][n]["n_touch"]), centre=True) for n in FS])
print(f"variant, latency at the centre of the crossing bin: {ci(lat_c)}; within (5.3+-0.53, 4.5+-0.45): {within(np.nanmean(lat_c), 5.3, 0.53)}, {within(np.nanmedian(lat_c), 4.5, 0.45)}")

**Result: partially reproduced.** FS neurons fire 1.83 +- 0.975 spikes per touch (median 1.95, n = 26), within 1.9 +- 0.2 and 1.9 +- 0.19. 25 neurons pass the latency exclusion (24 +- 2.4). The latency averages 5.92 ms (median 4 ms), outside 5.3 +- 0.53 and 4.5 +- 0.45. Read at the bin centre, the median is 4.5 ms (within) and the mean 6.42 ms (outside). Three of the five values hold under either edge.

## fs-whisking-count

> Most L4 FS neurons (23 of 26) significantly increase their spike rate during whisker movement
> without touch, and the rest (3 of 26) decrease it.

In [ ]:
for mg in "ab":
    for rd in "ab":
        sg = [msig(S[mg][n], rd)[0] for n in FS]
        up, dn, ns = sg.count(1), sg.count(-1), sg.count(0)
        print(f"margin ({mg}), samples ({rd}): increase {up}, decrease {dn}, no significant change {ns}; within (23+-2.3, 3+-1): {within(up, 23, 2.3) and within(dn, 3, 1)}")

**Result: reproduced.** 21 or 22 of 26 FS neurons increase significantly with whisking and 3 decrease, within 23 +- 2.3 and 3 +- 1 under all four readings.

## fs-whisking-rate

> With whisking the L4 FS spike rate doubles (to 21.1 +- 15.8 spikes/s, median 14.6, n = 26, P =
> 2.7e-5).

In [ ]:
for mg in "ab":
    for rr in "ab":
        nw = np.array([S[mg][n][f"nonw_{rr}"] for n in FS])
        wh = np.array([S[mg][n][f"whisk_{rr}"] for n in FS])
        p = stats.wilcoxon(wh, nw).pvalue
        ratio = wh.mean() / nw.mean()
        ok = within(wh.mean(), 21.1, 3.1) and within(np.median(wh), 14.6, 1.46) and p < 0.05 and wh.mean() > nw.mean() and 1.8 <= ratio <= 2.5
        print(f"margin ({mg}), rate ({rr}): whisking {ci(wh)}; nonwhisking mean {nw.mean():.2f}; ratio {ratio:.2f}; signed-rank p = {p:.2g}; all parts hold: {ok}")

**Result: reproduced.** The whisking rate is 21.7 +- 16.3 spikes/s (median 14.2), against 9.43 nonwhisking: a ratio of 2.30, signed-rank p = 2.3e-5, n = 26. The means, the medians, the test and the ratio (between 1.8 and 2.5) hold under all four readings (ratios 2.26 to 2.30).

## fs-phase-coupling

> L4 FS spikes are coupled to whisking phase; the phase relation is fit by a single sinusoid in 18
> neurons and needs a second harmonic in 7 (n = 25).

A neuron needs the second harmonic when the r2 of the sum is at least 1.5 times that of the
single sinusoid (Supplementary Fig. 7g legend).

In [ ]:
for nb in (8, 16):
    r1 = np.array([S["a"][n][f"phase{nb}"][0] for n in FS])
    r2h = np.array([S["a"][n][f"phase{nb}h"][0] for n in FS])
    harm = r2h >= 1.5 * r1
    print(f"{nb} bins: single {int((~harm).sum())}, with second harmonic {int(harm.sum())} (n = {len(FS)}); within (18+-1.8, 7+-1): {within((~harm).sum(), 18, 1.8) and within(harm.sum(), 7, 1)}; "
          f"single-sinusoid r2 median {np.median(r1):.2f}, r2 > 0.25 in {int((r1 > 0.25).sum())}")

**Result: partially reproduced.** FS spikes are coupled to whisking phase: the single sinusoid has r2 > 0.25 in 25 of 26 neurons (8 bins). The split differs from the paper's. A second harmonic raises r2 by half or more in 4 neurons (8 bins) or 5 (16 bins), leaving 22 or 21 with a single sinusoid, against 18 and 7. The counts are outside tolerance under both binnings.

## l4e-spikes-touch-selective

> L4 excitatory neurons respond mostly to touch (0.3 +- 0.4 spikes per touch, median 0.1, n =
> 85) and only weakly to whisker movement, with no significant change between nonwhisking (0.4 +-
> 0.6 spikes/s, median 0.1) and whisking (0.6 +- 0.9 spikes/s, median 0.2; P = 0.20).

In [ ]:
exp = [(0.3, 0.043), (0.1, 0.01), (0.4, 0.065), (0.1, 0.01), (0.6, 0.098), (0.2, 0.02)]
for mg in "ab":
    for rr in "ab":
        spt = np.array([S[mg][n]["spt"] for n in L4E85])
        nw = np.array([S[mg][n][f"nonw_{rr}"] for n in L4E85])
        wh = np.array([S[mg][n][f"whisk_{rr}"] for n in L4E85])
        ok = np.isfinite(spt)
        vals = [np.nanmean(spt), np.nanmedian(spt), nw.mean(), np.median(nw), wh.mean(), np.median(wh)]
        p = stats.wilcoxon(wh, nw).pvalue
        print(f"margin ({mg}), rate ({rr}): spikes/touch {ci(spt)}; nonwhisking {ci(nw)}; whisking {ci(wh)}; "
              f"signed-rank p = {p:.3g}; difference mean {np.mean(wh - nw):.3f}, median {np.median(wh - nw):.3f}; "
              f"within: {[within(v, e, t) for v, (e, t) in zip(vals, exp)]}")
print("by recording type (margin a, rate a): whole-cell spikes/touch", ci([S['a'][n]['spt'] for n in L4EWC]), "; cell-attached", ci([S['a'][n]['spt'] for n in names('L4E_ca')]))
spt = np.array([S_cfg[n]["spt"] for n in L4E85])
nw = np.array([S_cfg[n]["nonw_a"] for n in L4E85])
wh = np.array([S_cfg[n]["whisk_a"] for n in L4E85])
ok = np.isfinite(nw) & np.isfinite(wh)
print(f"variant, principal whisker by the configuration (margin a, rate a): spikes/touch {ci(spt)}; nonwhisking {ci(nw)}; whisking {ci(wh)}; "
      f"signed-rank p = {stats.wilcoxon(wh[ok], nw[ok]).pvalue:.3g}; within: {[within(v, e, t) for v, (e, t) in zip([np.nanmean(spt), np.nanmedian(spt), np.nanmean(nw), np.nanmedian(nw), np.nanmean(wh), np.nanmedian(wh)], exp)]}")

**Result: not reproduced.** The claim of no whisking effect fails its test. The signed-rank test of whisking against nonwhisking rates over the 85 neurons gives p = 0.020 to 0.027 under all four readings (mean difference 0.29 to 0.33 spikes/s, median difference 0). The nonwhisking mean (0.428) and both medians (0.09 and 0.21) are within tolerance. The whisking mean (0.718 to 0.759) is not, nor are spikes per touch (mean 0.20, median 0.04, against 0.3 and 0.1). Whole-cell neurons fire fewer spikes per touch (0.096) than cell-attached ones (0.28). With the principal whisker taken from the whisker configuration, the test gives p = 0.0061 and spikes per touch fall to 0.172, so the outcome does not rest on that choice.

## ion-cut-behavior

> After ION transection mice still whisk but perform the task at chance (48% correct).

In [ ]:
ion = [c for c in cells if ssc7.group(c) in ("L4E_ion", "FS_ion")]
per = [behav(c, c["good"]) for c in ion]
pooled = np.vstack(per)[:, 1].mean()
mean_cells = np.mean([p[:, 1].mean() for p in per])
mice = pd.Series([p[:, 1].mean() for p in per], index=[c["meta"]["subject"] for c in ion])
print(f"ION-cut recordings: {len(ion)} cells, {mice.index.nunique()} mice, {sum(len(p) for p in per)} typed trials")
print(f"(a) pooled over trials: {pooled:.3f}; (b) mean over cells: {mean_cells:.3f} +- {np.std([p[:, 1].mean() for p in per], ddof=1):.3f}; within 0.48+-0.048: {within(pooled, 0.48, 0.048)}, {within(mean_cells, 0.48, 0.048)}")
print("by mouse:", mice.groupby(level=0).mean().round(3).to_dict())
amp_ion = np.concatenate([np.nanmean(np.vstack([d["amp"] for d in S["a"][ssc7.cell_name(c)]["ep"].values()]), 1) for c in ion])
amp_ctl = np.concatenate([np.nanmean(np.vstack([d["amp"] for d in S["a"][n]["ep"].values()]), 1) for n in L4E85 + FS])
print(f"mean whisking amplitude per trial (degrees): ION cut median {np.median(amp_ion):.2f}, control median {np.median(amp_ctl):.2f}")

**Result: reproduced.** After ION cut, 0.492 of 1,335 trials are correct (pooled), and the mean over 19 cells is 0.510. Both are within 0.48 +- 0.048. The median whisking amplitude per trial is 1.60 degrees, against 2.56 in control recordings, so the mice still whisk.

## ion-cut-fs-touch

> After ION transection touch does not evoke spikes in L4 FS interneurons (n = 10).

In [ ]:
x = np.array([S["a"][n]["spt"] for n in FSION])
print("spikes per touch, ION cut:", ci(x), "; values", x.round(3))
p = stats.wilcoxon(x).pvalue
print(f"signed-rank against zero: p = {p:.3g}; control FS mean {np.mean([S['a'][n]['spt'] for n in FS]):.2f}")

**Result: reproduced.** After ION cut, FS spikes per touch are 0.061 +- 0.115 (median 0.040, n = 10), and the signed-rank test against zero is not significant (p = 0.098). In control FS neurons the mean is 1.83.

## ion-cut-fs-whisking

> After ION transection whisking-related activation of L4 FS interneurons is absent in most cells
> (P = 0.80, n = 10).

In [ ]:
for mg in "ab":
    for rr in "ab":
        nw = np.array([S[mg][n][f"nonw_{rr}"] for n in FSION])
        wh = np.array([S[mg][n][f"whisk_{rr}"] for n in FSION])
        p = stats.wilcoxon(wh, nw).pvalue
        print(f"margin ({mg}), rate ({rr}): nonwhisking {nw.mean():.2f}, whisking {wh.mean():.2f}; signed-rank p = {p:.3g}; median ratio {np.median(wh / nw):.2f}")
for rd in "ab":
    sg = [msig(S["a"][n], rd)[0] for n in FSION]
    print(f"M-SIG ({rd}): increase {sg.count(1)}, decrease {sg.count(-1)}, no change {sg.count(0)} of {len(FSION)}")
print("control FS median whisking/nonwhisking ratio:", np.median([S['a'][n]['whisk_a'] / S['a'][n]['nonw_a'] for n in FS]).round(2))

**Result: reproduced.** After ION cut the whisking and nonwhisking rates of the 10 FS neurons do not differ (signed-rank p = 0.85 under reading a; 0.63 to 0.85 under all four). The median whisking to nonwhisking ratio is 0.96, against 2.71 in controls. By M-SIG, 3 of 10 neurons increase significantly under either reading, so the activation is absent in most cells.

## ion-cut-fs-phase

> After ION transection phase-locking of L4 FS spikes to whisker movement is reduced but not
> entirely abolished.

In [ ]:
for nb in (8, 16):
    rc = np.array([S["a"][n][f"phase{nb}"][0] for n in FS])
    ri = np.array([S["a"][n][f"phase{nb}"][0] for n in FSION])
    p = stats.ranksums(ri, rc).pvalue
    print(f"{nb} bins: r2 control median {np.median(rc):.2f} (n={len(rc)}), ION cut median {np.median(ri):.2f} (n={len(ri)}); rank-sum p = {p:.3g}; "
          f"ION-cut neurons with r2 > 0.25: {int((ri > 0.25).sum())}; holds: {p < 0.05 and np.median(ri) < np.median(rc) and (ri > 0.25).any()}")

**Result: reproduced.** The r2 of the spike-phase fit falls from a median of 0.82 in control FS neurons to 0.48 after ION cut (8 bins, rank-sum p = 0.013), or from 0.70 to 0.30 (16 bins, p = 0.0038). Nine (8 bins) or six (16 bins) of the 10 ION-cut neurons keep r2 > 0.25.

## l4e-example-e1

> The example L4 spiny stellate cell E No. 1 rests at -76 mV, far from its spike threshold
> (-45.1 +- 3.7 mV); touch evokes large (14.4 +- 8.2 mV), short-latency (4.7 ms), transient
> (24.5 ms at half max) depolarizations with 0 to 7 spikes per touch, while whisking without
> touch causes a smaller sustained depolarization (6.7 +- 4.0 mV) with little increase in spike
> rate (0.4 spikes/s).

E No. 1 is JY0861 (the cell list's ExampleCells sheet, Figure 2 "# 1: 'JY0861'", and the file's
unit description "L4 excitatory neuron: # 1").

In [ ]:
v = V["JY0861AAAA"]
print("unit description:", NAME["JY0861AAAA"]["meta"]["unit_description"], "; control trials", v["n_trials"], "; touches", v["n_touch"], "; onsets", v["n_onsets"])
cnt = np.array(v["spk_counts_touch"])
rows = [("resting Vm (a) nonwhisking mean", v["rest_a"], -76, 7.6), ("resting Vm (b) median outside touch", v["rest_b"], -76, 7.6),
        ("spike threshold, mean over spikes", v["thr_mean"], -45.1, 4.51),
        ("touch PSP amplitude (peak 0-50 ms of the average)", v["amp50"], 14.4, 1.44),
        ("touch PSP amplitude (peak 0-25 ms)", v["amp25"], 14.4, 1.44),
        ("touch PSP latency (ms)", v["lat"], 4.7, 0.47), ("half width (ms)", v["hw"], 24.5, 2.45),
        ("whisking depolarization (a) onset 100-300 ms minus 0-200 ms before", v["wdep_a"], 6.7, 0.67),
        ("whisking depolarization (b) whisking minus nonwhisking epochs", v["wdep_b"], 6.7, 0.67),
        ("spike rate (a) whisking minus nonwhisking", v["whisk_rate"] - v["nonw_rate"], 0.4, 0.04),
        ("spike rate (b) whisking", v["whisk_rate"], 0.4, 0.04)]
for r in rows:
    print(f"{r[0]}: {r[1]:.2f} (paper {r[2]}, tolerance {r[3]}) within: {within(r[1], r[2], r[3])}")
print(f"threshold s.d. over spikes {np.std(ssc7.prep_vm(NAME['JY0861AAAA'])['thr']):.2f} mV; amplitude s.d. over touches {v['amp_sd']:.2f}; whisking change s.d. over onsets {v['wdep_a_sd']:.2f}")
print("spikes per touch 0-50 ms: range", cnt.min(), "to", cnt.max(), "; distribution", np.bincount(cnt).tolist())

**Result: reproduced.** For JY0861 (E No. 1), all seven values are within tolerance under readings (a). Resting Vm is -76.0 mV, spike threshold -44.87 mV, touch PSP 15.50 mV, latency 4.8 ms, half width 23.5 ms, whisking depolarization 6.10 mV (onsets) and whisking minus nonwhisking rate 0.41 spikes/s. Spikes per touch range from 0 to 7. The readings not chosen differ: 5.52 mV for the epoch-based whisking change and 0.55 spikes/s for the whisking rate. The spread over spikes and touches is larger than the paper's (threshold s.d. 8.91 against 3.7 mV).

## l4e-example-e24-e32

> In the more depolarized example neurons, whisking causes a small depolarization (1.0 +- 1.4 mV,
> E No. 24) or a hyperpolarization (-1.8 +- 2.7 mV, E No. 32).

E No. 24 is JY0520 and E No. 32 is JY1008 (ExampleCells sheet; unit descriptions "#24", "#32").

In [ ]:
for n, e in (("JY0520AAAC", 1.0), ("JY1008AAAA", -1.8)):
    v = V[n]
    t = abs(e) * 0.1
    print(f"{n} ({NAME[n]['meta']['unit_description']}): paper {e} +- {t:.2f}; resting Vm {v['rest_a']:.1f}; onsets n = {v['n_onsets']}")
    for k, lab in (("wdep_a", "(a) onsets, 250 ms rise"), ("wdep_a_rise100", "(a) onsets, 100 ms rise"), ("wdep_b", "(b) epochs")):
        print(f"   {lab}: {v[k]:.2f}; within {within(v[k], e, t)}; sign as the paper's {np.sign(v[k]) == np.sign(e)}")

**Result: reproduced.** Both neurons have the paper's sign under every reading. With the onset measure and the 100 ms rise of M-EPOCH reading (b), E No. 24 (JY0520) changes by +1.07 mV and E No. 32 (JY1008) by -1.75 mV, both within tolerance (1.0 +- 0.1, -1.8 +- 0.18). With the 250 ms rise, E No. 24 gives +1.29 (outside) and E No. 32 -1.65 (within); the epoch measure gives +1.47 and -2.04, both outside. The criterion accepts one reading. The reading that matches here is not the one that matches l4e-whisking-vm (the 250 ms rise), so no single reading matches both claims.

## l4e-resting-vm

> The resting membrane potential of L4 excitatory neurons spans a large range, -76 to -45 mV
> (mean -63 +- 8 mV, median -64 mV, n = 37).

In [ ]:
exp = [(-76, 7.6), (-45, 4.5), (-63, 6.3), (-64, 6.4)]
for rd in "ab":
    x = np.array([V[n][f"rest_{rd}"] for n in L4EWC])
    vals = [x.min(), x.max(), x.mean(), np.median(x)]
    print(f"reading ({rd}): min {x.min():.1f}, max {x.max():.1f}, {ci(x)}; within: {[within(v_, e, t) for v_, (e, t) in zip(vals, exp)]}")

**Result: reproduced.** Over the 37 whole-cell L4 E neurons, resting Vm spans -76.0 to -44.3 mV, with mean -62.7 +- 7.4 and median -63.4 mV (reading a). All four values are within tolerance under both readings.

## l4e-touch-psp

> Touch evokes short-latency depolarizations in every L4 excitatory neuron (latency 4.8 +- 0.8
> ms, median 4.7; amplitude 6.0 +- 2.9 mV, median 5.7; n = 37).

In [ ]:
lat = np.array([V[n]["lat"] for n in L4EWC])
for win in ("amp50", "amp25"):
    amp = np.array([V[n][win] for n in L4EWC])
    vals = [np.nanmean(lat), np.nanmedian(lat), amp.mean(), np.median(amp)]
    exp = [(4.8, 0.48), (4.7, 0.47), (6.0, 0.6), (5.7, 0.57)]
    print(f"peak window {win}: latency {ci(lat)}; amplitude {ci(amp)}; all amplitudes positive: {bool((amp > 0).all())}; within: {[within(v_, e, t) for v_, (e, t) in zip(vals, exp)]}")
print("touches per neuron: median", np.median([V[n]["n_touch"] for n in L4EWC]), "min", min(V[n]["n_touch"] for n in L4EWC))
print("neurons without a latency crossing:", [n for n in L4EWC if not np.isfinite(V[n]['lat'])])
print("latencies (ms):", np.round(sorted(lat[np.isfinite(lat)]), 1).tolist())
lat_cfg = np.array([V_cfg[n]["lat"] for n in L4EWC])
amp_cfg = np.array([V_cfg[n]["amp50"] for n in L4EWC])
vals = [np.nanmean(lat_cfg), np.nanmedian(lat_cfg), np.nanmean(amp_cfg), np.nanmedian(amp_cfg)]
print(f"variant, principal whisker by the configuration (peak 0-50 ms): latency {ci(lat_cfg)}; amplitude {ci(amp_cfg)}; all positive {bool((amp_cfg[np.isfinite(amp_cfg)] > 0).all())}; "
      f"within: {[within(v_, e, t) for v_, (e, t) in zip(vals, exp)]}")

**Result: partially reproduced.** Every one of the 37 neurons has a positive touch PSP. The amplitude has a mean of 5.26 mV, outside 6.0 +- 0.6, and a median of 5.4 mV, within 5.7 +- 0.57. The latency has a median of 5.0 ms, within 4.7 +- 0.47. Its mean is 6.03 ms (n = 35), outside 4.8 +- 0.48, raised by a few long latencies in neurons with small PSPs (up to 22.1 ms). Two of four values hold under both peak windows. With the principal whisker taken from the whisker configuration, the amplitude falls to 4.18 mV (median 3.7) and only the latency median (4.8 ms) holds.

## l4e-whisking-vm

> Whisking induces only a small depolarization or hyperpolarization in L4 excitatory neurons
> (1.2 +- 2.1 mV, median 1.2 mV).

In [ ]:
for k in ("wdep_a", "wdep_a_rise100", "wdep_b"):
    x = np.array([V[n][k] for n in L4EWC])
    print(f"{k}: {ci(x)}; range {np.nanmin(x):.2f} to {np.nanmax(x):.2f}; within (1.2+-0.35, 1.2+-0.12): {within(np.nanmean(x), 1.2, 0.35)}, {within(np.nanmedian(x), 1.2, 0.12)}")

**Result: reproduced.** With the onset measure (reading a), whisking changes Vm by 0.994 +- 2.03 mV (median 1.16, range -6.49 to 6.10, n = 37), within 1.2 +- 0.35 and 1.2 +- 0.12. The epoch measure (b) gives 1.37 (median 1.35, outside), and the 100 ms rise rule gives 0.677 (median 0.851, outside).

## l4e-distance-threshold

> Touch moves the membrane potential of L4 excitatory neurons much closer to spike threshold than
> whisking does (shift 6.1 +- 3.2 mV against 1.1 +- 2.0 mV, P = 1e-7, Wilcoxon signed-rank).

In [ ]:
for thr, sfx in (("per-trial line (M-VM)", ""), ("line held outside the trials with spikes (variant)", "_clamp"), ("one mean threshold per neuron (variant)", "_const")):
    for rd in "ab":
        w = np.array([V[n][f"shift_whisk_{rd}{sfx}"] for n in L4EWC])
        t = np.array([V[n][f"shift_touch_{rd}{sfx}"] for n in L4EWC])
        ok = np.isfinite(w) & np.isfinite(t)
        p = stats.wilcoxon(t[ok], w[ok]).pvalue
        print(f"{thr}, spikes {'removed' if rd == 'a' else 'kept'} ({rd}): whisking shift {ci(w)}; touch shift {ci(t)}; signed-rank p = {p:.2g}; "
              f"within (1.1+-0.33, 6.1+-0.61): {within(np.nanmean(w), 1.1, 0.33)}, {within(np.nanmean(t), 6.1, 0.61)}")
        print(f"   medians: whisking {np.nanmedian(w):.2f}, touch {np.nanmedian(t):.2f}; |shift| > 15 mV: {int(np.sum(np.abs(w[ok]) > 15) + np.sum(np.abs(t[ok]) > 15))}")
silent = [n for n in L4EWC if V[n]["n_spikes_thr"] == 0]
print("neurons with no detected spike threshold (the -42 mV default):", len(silent))
TS = pd.DataFrame([dict(name=n, thresholds=V[n]["n_spikes_thr"], slope_mV_per_trial=round(V[n]["thr_slope"], 3), first_trial=V[n]["thr_trial_range"][0],
                        last_trial=V[n]["thr_trial_range"][1], n_trials=len(NAME[n]["vm"]), shift_whisk_line=round(V[n]["shift_whisk_a"], 2),
                        shift_whisk_clamp=round(V[n]["shift_whisk_a_clamp"], 2), shift_whisk_const=round(V[n]["shift_whisk_a_const"], 2),
                        shift_touch_line=round(V[n]["shift_touch_a"], 2), shift_touch_const=round(V[n]["shift_touch_a_const"], 2)) for n in L4EWC])
print("threshold line per neuron (spikes removed):")
print(TS.to_string(index=False))
w = np.array([V_cfg[n]["shift_whisk_a"] for n in L4EWC])
t = np.array([V_cfg[n]["shift_touch_a"] for n in L4EWC])
ok = np.isfinite(w) & np.isfinite(t)
print(f"variant, principal whisker by the configuration (line, removed): whisking {ci(w)}; touch {ci(t)}; signed-rank p = {stats.wilcoxon(t[ok], w[ok]).pvalue:.2g}")

**Result: partially reproduced.** Touch moves Vm closer to threshold than whisking does: medians 5.42 against 0.87 mV, signed-rank p = 2.9e-11. With the per-trial threshold line of M-VM, the touch mean (5.74 mV, spikes removed; 5.81 kept) is within 6.1 +- 0.61, but the whisking mean (-0.236 +- 5.89) is outside 1.1 +- 0.33. Three shifts exceed 15 mV. The largest whisking shift, -28.9 mV in JY0947, comes from a line through two thresholds in trials 126 and 129 extrapolated over 150 trials. With the line held at its end values outside the trials with spikes (variant), the means are 0.767 and 5.34 mV, both outside. Only with one mean threshold per neuron, which is not the paper's method, are both means within tolerance (0.784 and 5.52 mV, spikes kept). 11 neurons have no detected threshold and take the -42 mV default. The principal-whisker variant gives the same test result (p = 6.1e-6).

## l4e-reversal-potential

> The effective reversal potential of the touch response in L4 excitatory neurons is more
> depolarized than resting Vm, while that of the whisking response is close to resting Vm and
> more hyperpolarized than that of touch.

In [ ]:
def reversal(pre, post):
    if len(pre) < 5:
        return np.nan, np.nan
    sl, ic = np.polyfit(pre, post - pre, 1)
    return (-ic / sl if sl < 0 else np.nan), sl


rv = []
for n in L4EWC:
    v = V[n]
    et, st_ = reversal(v.get("touch_pre", np.array([])), v.get("touch_post", np.array([])))
    oc = v["onset_pre_post"]
    ew, sw = reversal(oc[:, 0], oc[:, 1]) if len(oc) else (np.nan, np.nan)
    rv.append(dict(name=n, rest=v["rest_a"], e_touch=et, slope_touch=st_, e_whisk=ew, slope_whisk=sw))
rv = pd.DataFrame(rv)
print("non-negative slopes (no crossing), touch:", int((rv.slope_touch >= 0).sum()), ", whisking:", int((rv.slope_whisk >= 0).sum()))
ok = rv.e_touch.notna() & rv.e_whisk.notna()
p = stats.wilcoxon(rv.e_touch[ok], rv.e_whisk[ok]).pvalue
print(f"(a) per neuron, n = {ok.sum()}: touch reversal median {rv.e_touch[ok].median():.1f} mV, whisking reversal median {rv.e_whisk[ok].median():.1f}, signed-rank p = {p:.2g}")
print(f"   touch reversal above resting Vm in {int((rv.e_touch > rv.rest).sum())} of {int(rv.e_touch.notna().sum())}; median whisking reversal minus median resting Vm "
      f"{rv.e_whisk.median() - rv.rest.median():.1f} mV (rest median {rv.rest.median():.1f})")
# (b) pooled over neurons
tp = np.concatenate([V[n]["touch_pre"] for n in L4EWC if "touch_pre" in V[n]])
tq = np.concatenate([V[n]["touch_post"] for n in L4EWC if "touch_post" in V[n]])
oc = np.vstack([V[n]["onset_pre_post"] for n in L4EWC if len(V[n]["onset_pre_post"])])
print(f"(b) pooled: touch reversal {reversal(tp, tq)[0]:.1f} mV ({len(tp)} touches), whisking reversal {reversal(oc[:, 0], oc[:, 1])[0]:.1f} mV ({len(oc)} onsets)")
print(rv.round(2).to_string(index=False))

**Result: reproduced.** Per neuron (n = 35), the touch reversal potential (median -49.8 mV) exceeds the whisking reversal (median -58.7 mV), signed-rank p = 2.6e-5. The touch reversal lies above resting Vm in 34 of 35 neurons. The median whisking reversal lies 4.9 mV above the median resting Vm (-63.4 mV), just inside the 5 mV margin the criterion fixes. The pooled fit gives the same picture (touch -45.4, whisking -57.6 mV). The margin is close, so this part could change with small differences in the onset rules.

## l4e-hf-fluctuations

> Whisking enhances membrane potential fluctuations of L4 excitatory neurons at high frequencies
> (above 10 Hz).

In [ ]:
def band_power(spec, lo, hi):
    if spec is None:
        return np.nan
    f, m, l_, h_ = spec
    k = (f >= lo) & (f <= hi)
    return m[k].mean()


def above_ci(v, lo, hi):
    """Fraction of the band where whisking power lies above the nonwhisking upper 95% bound."""
    if v["spec_nonw"] is None or v["spec_whisk"] is None:
        return np.nan
    f, mn, ln, hn = v["spec_nonw"]
    _, mw, lw, hw = v["spec_whisk"]
    k = (f >= lo) & (f <= hi)
    return np.mean(mw[k] > hn[k])


def spectra_test(group, label):
    out = {}
    for rd, (lo, hi) in (("a", (10, 100)), ("b", (10, 500))):
        pn = np.array([band_power(V[n]["spec_nonw"], lo, hi) for n in group])
        pw_ = np.array([band_power(V[n]["spec_whisk"], lo, hi) for n in group])
        ok = np.isfinite(pn) & np.isfinite(pw_)
        p = stats.wilcoxon(pw_[ok], pn[ok]).pvalue
        frac = np.array([above_ci(V[n], lo, hi) for n in group])
        out[rd] = (p, np.median(pw_[ok] / pn[ok]), ok.sum(), frac)
        print(f"{label}, band ({rd}) {lo}-{hi} Hz: n = {ok.sum()}, whisking/nonwhisking power median ratio {np.median(pw_[ok] / pn[ok]):.2f}, "
              f"higher in {int(np.sum(pw_[ok] > pn[ok]))}; signed-rank p = {p:.2g}; neurons with whisking above the nonwhisking 95% bound over most of the band: {int(np.nansum(frac > 0.5))}")
    return out


segs = pd.DataFrame([dict(name=n, nonw=V[n]["n_seg"][0], whisk=V[n]["n_seg"][1]) for n in L4EWC])
print("500 ms segments per neuron: nonwhisking median", segs.nonw.median(), "; whisking median", segs.whisk.median(), "; neurons with no whisking segment:", int((segs.whisk == 0).sum()))
hf_l4e = spectra_test(L4EWC, "L4E")

**Result: reproduced.** Over the 36 neurons with both kinds of segment, whisking power exceeds nonwhisking power by a signed-rank test in band (a), 10 to 100 Hz (median ratio 1.17, p = 0.046), and in band (b), 10 to 500 Hz (ratio 1.18, p = 0.030). Whisking power lies above the nonwhisking 95% bound over most of the band in 24 (a) and 30 (b) neurons. The p-value in band (a) is close to 0.05.

## fs-touch-vm

> Touch depolarizes L4 FS interneurons (7.0 +- 5.2 mV, median 6.7 mV, n = 7) with short onset
> latency (4.0 +- 0.6 ms, median 4.2 ms).

In [ ]:
lat = np.array([V[n]["lat"] for n in FSWC])
exp = [(7.0, 1.97), (6.7, 0.67), (4.0, 0.4), (4.2, 0.42)]
for win in ("amp50", "amp25"):
    amp = np.array([V[n][win] for n in FSWC])
    vals = [amp.mean(), np.median(amp), np.nanmean(lat), np.nanmedian(lat)]
    print(f"{win}: amplitude {ci(amp)}; latency {ci(lat)}; within: {[within(v_, e, t) for v_, (e, t) in zip(vals, exp)]}")
print("per neuron:", {n: (round(V[n]['amp50'], 2), round(V[n]['lat'], 1), V[n]['n_touch']) for n in FSWC})
lat_all = np.array([V_all[n]["lat"] for n in FSWC])
amp_all = np.array([V_all[n]["amp50"] for n in FSWC])
vals = [amp_all.mean(), np.median(amp_all), np.nanmean(lat_all), np.nanmedian(lat_all)]
print(f"variant, every control trial (JY1139's trials far from its usual Vm kept): amplitude {ci(amp_all)}; latency {ci(lat_all)}; within: {[within(v_, e, t) for v_, (e, t) in zip(vals, exp)]}")

**Result: reproduced.** The seven whole-cell FS neurons depolarize by 6.8 +- 1.93 mV after touch (median 6.97), with latency 3.67 +- 1.18 ms (median 3.9). All four values are within tolerance. Keeping JY1139's drifting trials (variant) gives 6.74, 6.97, 3.69 and 4 ms, also all within.

## fs-whisking-vm

> After whisking onset the membrane potential of L4 FS neurons depolarizes toward spike threshold
> (2.7 +- 2.1 mV, median 2.4 mV, n = 7).

In [ ]:
for lab, VV in (("trials far from the usual Vm left out", V), ("every control trial (variant)", V_all)):
    print(lab + ":")
    for k in ("wdep_a", "wdep_a_rise100", "wdep_b", "shift_whisk_a", "shift_whisk_a_clamp", "shift_whisk_a_const"):
        x = np.array([VV[n][k] for n in FSWC])
        print(f"   {k}: {ci(x)}; within (2.7+-0.79, 2.4+-0.24): {within(np.nanmean(x), 2.7, 0.79)}, {within(np.nanmedian(x), 2.4, 0.24)}")
print("per neuron, wdep_a:", {n: round(V[n]["wdep_a"], 2) for n in FSWC})

**Result: partially reproduced.** FS neurons depolarize after whisking onset. The onset measure gives a mean of 2.02 mV, within 2.7 +- 0.79, and a median of 1.52, outside 2.4 +- 0.24. The epoch measure gives 2.43, median 2.0. The 100 ms rise gives 1.54 (median 0.867). The distance-to-threshold reading (c) gives 0.976 mV (1.29 with one threshold per neuron). Under no reading is the median within tolerance, with or without JY1139's drifting trials (onsets 2.0 and 1.49 with them).

## fs-hf-fluctuations

> In L4 FS interneurons, membrane potential fluctuations at high frequencies (above 10 Hz)
> increase with whisking in some cells but not in others (n = 7 whole-cell).

In [ ]:
print("500 ms segments (nonwhisking, whisking):", {n: V[n]["n_seg"] for n in FSWC})
hf_fs = spectra_test(FSWC, "FS")
for rd in "ab":
    frac = hf_fs[rd][3]
    k = int(np.nansum(frac > 0.5))
    print(f"band ({rd}): fraction of band above the bound per neuron {np.round(frac, 2).tolist()}; neurons showing the increase {k}; 1 to 6 of 7: {1 <= k <= 6}")
# variant: every control trial, JY1139's trials far from its usual Vm kept
V_keep = V
V = {**V, **V_all}
print("variant, every control trial:")
hf_fs_all = spectra_test(FSWC, "FS")
V = V_keep
# Chance level of the per-neuron rule: nonwhisking segments split at random into a pseudo-whisking
# set (as many segments as the neuron has whisking segments) and the rest; how often does the
# pseudo-whisking power lie above the bound over most of the band?
rng = np.random.default_rng(9)
chance = {"a": [], "b": []}
for n in FSWC:
    c = NAME[n]
    P_ = ssc7.prep_vm(c)
    ep_ = ssc7.cell_epochs(c, ssc7.control_trials(c) & ssc7.stable_trials(c))
    sn = ssc7.segments_500ms(c, P_["vm"], ep_, "nonw")
    k = V[n]["n_seg"][1]
    if k == 0 or len(sn) < k + 3:
        for rd in "ab":
            chance[rd].append(np.nan)
        continue
    hits = {"a": 0, "b": 0}
    for _ in range(200):
        perm = rng.permutation(len(sn))
        fake = dict(spec_whisk=ssc7.multitaper(sn[perm[:k]]), spec_nonw=ssc7.multitaper(sn[perm[k:]]))
        for rd, (lo, hi) in (("a", (10, 100)), ("b", (10, 500))):
            hits[rd] += above_ci(fake, lo, hi) > 0.5
    for rd in "ab":
        chance[rd].append(hits[rd] / 200)
for rd in "ab":
    print(f"band ({rd}): chance that a neuron shows the increase with no whisking effect, per neuron {np.round(chance[rd], 2).tolist()}; expected count {np.nansum(chance[rd]):.1f} of 7")

**Result: reproduced.** Whisking power lies above the nonwhisking 95% jackknife bound over most of the band in 6 of 7 FS neurons, for 10 to 100 Hz and for 10 to 500 Hz (5 and 6 with JY1139's drifting trials kept). Both counts are between 1 and 6, so some cells increase and one does not. The test is weak: four neurons have 1 to 3 whisking segments. With nonwhisking segments split at random into pseudo-whisking and nonwhisking sets, a neuron passes the rule by chance with probability 0.02 to 0.22, about 1.0 (a) and 0.7 (b) of the 7 neurons, so 6 of 7 is well above chance.

## ion-cut-l4e-touch

> After ION transection L4 excitatory neurons do not depolarize after touch.

In [ ]:
amp = np.array([V[n]["amp50"] for n in L4EION])
print("touches per ION-cut neuron:", [V[n]["n_touch"] for n in L4EION])
ok = np.isfinite(amp)
p = stats.wilcoxon(amp[ok]).pvalue
print(f"touch PSP amplitude (peak 0-50 ms minus 0-5 ms before), ION cut: {ci(amp)}; signed-rank against zero p = {p:.3g}; control L4E mean {np.mean([V[n]['amp50'] for n in L4EWC]):.2f} mV")
amp25 = np.array([V[n]["amp25"] for n in L4EION])
print(f"variant, peak 0-25 ms: {ci(amp25)}")
# reference: the same peak measure on random times (no touch), which is positive by construction
rng = np.random.default_rng(3)
null = []
for n in L4EION:
    c = NAME[n]
    P = ssc7.prep_vm(c)
    tr = np.flatnonzero(ssc7.control_trials(c))
    tt = np.column_stack([rng.choice(tr, 200), rng.uniform(0.5, 4.5, 200)])
    X = ssc7.touch_vm(c, P["vm"], tt)
    null.append(ssc7.psp_amplitude(X.mean(0))[0] if len(X) else np.nan)
print(f"same measure at 200 random times per neuron: {ci(null)}; touch minus random-time amplitude, signed-rank p = {stats.wilcoxon(amp[ok] - np.array(null)[ok]).pvalue:.3g}")

**Result: not reproduced.** The criterion's test finds an effect. The touch PSP amplitude of the 8 ION-cut neurons with touches is 0.321 +- 0.222 mV, positive in all, and the signed-rank test against zero gives p = 0.0078. The amplitude is a peak minus a baseline, so it is positive even without a response. At random times the same measure gives 0.208 mV, and touch does not differ from it (p = 0.148, variant). The amplitude is 0.32 mV against 5.26 mV in controls.

## ion-cut-l4e-whisking

> After ION transection whisking-related depolarization of L4 excitatory neurons is reduced (0.62
> +- 0.50 mV, n = 9, against 1.54 +- 0.79 mV in 15 control neurons with matched resting
> potential; P = 0.005, Wilcoxon rank-sum).

In [ ]:
rest_ion = np.array([V[n]["rest_a"] for n in L4EION])
rest_ctl = pd.Series({n: V[n]["rest_a"] for n in L4EWC})
match_a = rest_ctl[(rest_ctl >= rest_ion.min()) & (rest_ctl <= rest_ion.max())].index.tolist()
match_b = (rest_ctl - rest_ion.mean()).abs().sort_values().index[:15].tolist()
print(f"ION-cut resting Vm range {rest_ion.min():.1f} to {rest_ion.max():.1f}; controls in range (a): {len(match_a)} (paper 15)")
MATCH = {"a": match_a, "b": match_b}
for k in ("wdep_a", "wdep_b"):
    xi = np.array([V[n][k] for n in L4EION])
    for rd, grp in MATCH.items():
        xc = np.array([V[n][k] for n in grp])
        p = stats.ranksums(xi, xc).pvalue
        print(f"{k}, controls ({rd}): ION {xi.mean():.3f} +- {xi.std(ddof=1):.2f} (n={len(xi)}), control {xc.mean():.3f} +- {xc.std(ddof=1):.2f} (n={len(xc)}); rank-sum p = {p:.3g}; "
              f"within (0.62+-0.17, 1.54+-0.2): {within(xi.mean(), 0.62, 0.17)}, {within(xc.mean(), 1.54, 0.2)}; holds: {within(xi.mean(), 0.62, 0.17) and within(xc.mean(), 1.54, 0.2) and p < 0.05 and xi.mean() < xc.mean()}")

**Result: partially reproduced.** Whisking depolarization is lower after ION cut under every reading, but the control mean is just outside its tolerance under all of them. With the onset measure and controls whose resting Vm lies within the ION-cut range (reading a, 17 neurons against the paper's 15), it is 0.582 +- 0.35 mV after ION cut against 1.337 +- 0.98 mV in controls, rank-sum p = 0.033: the control mean lies 0.003 mV below the bound of 1.54 - 0.2 = 1.34. With the 15 nearest controls (b) the control mean is 1.282 and p = 0.060. With the epoch measure the control mean is 1.80 to 1.83 (p = 0.0047 and 0.0067). The effect is present and significant under three of the four readings; only the control mean misses, by a margin smaller than any rounding in the paper.

## ion-cut-l4e-phase

> After ION transection phase-locking of L4 excitatory Vm to whisker movement is abolished
> (coupling 0.12 +- 0.04 mV against 0.38 +- 0.31 mV in controls, P = 0.005).

In [ ]:
for nb in (8, 16):
    ci_ = np.array([V[n][f"vphase{nb}"][1] for n in L4EION])
    for rd, grp, tol in (("a", match_a, 0.08), ("a'", match_b, 0.08), ("b", L4EWC, 0.051)):
        cc = np.array([V[n][f"vphase{nb}"][1] for n in grp])
        p = stats.ranksums(ci_, cc).pvalue
        print(f"{nb} bins, controls ({rd}, n={len(cc)}): ION {ci_.mean():.3f} +- {ci_.std(ddof=1):.3f}, control {cc.mean():.3f} +- {cc.std(ddof=1):.3f} mV; rank-sum p = {p:.3g}; "
              f"within (0.12+-0.013, 0.38+-{tol}): {within(ci_.mean(), 0.12, 0.013)}, {within(cc.mean(), 0.38, tol)}")

**Result: reproduced.** The Vm phase-coupling strength is 0.111 +- 0.052 mV after ION cut against 0.304 +- 0.239 mV in the range-matched controls (8 bins, rank-sum p = 0.033). Both are within tolerance (0.12 +- 0.013, 0.38 +- 0.08). The 15 nearest controls and 16 bins also hold. Against all 37 controls (reading b) the control mean is 0.454, outside 0.38 +- 0.051, with p = 0.0069.

## ion-cut-l4e-spectra

> After ION transection the Vm fluctuations of L4 excitatory neurons do not differ between
> nonwhisking and whisking epochs.

In [ ]:
print("500 ms segments (nonwhisking, whisking):", {n: V[n]["n_seg"] for n in L4EION})
hf_ion = spectra_test(L4EION, "ION-cut L4E")

**Result: reproduced.** Over the 7 ION-cut neurons with both kinds of segment, whisking and nonwhisking power do not differ (10 to 100 Hz: median ratio 0.97, p = 0.47). The 10 to 500 Hz variant gives the same (ratio 0.96, p = 0.47). Two neurons have no whisking segment.

### Photostimulation trials

Each file stores one photostimulus series per light site (for example "VPM", "VPM and vS1",
"vS1" or "vM1", with its wavelength), sampled with the ephys. A trial's light is read from these
series: its sites, its power (the peak of the series, mW) and its onset and offset. The
experiments are those of the Online Methods, Optogenetic stimulation:

- thalamic photoinhibition: the VPM-only 473 nm series on and no other series ("For trials in
  which only VPM stimulation was applied, a galvanometer in the optic path moved the laser away
  from the craniotomy");
- eNpHR suppression of PV interneurons: the 594 nm series on ("constant over time");
- direct cortical GABAergic activation: the vS1 473 nm series on alone, in VGAT-ChR2 mice.

Control trials are good trials without a stimulus flag and without light. Light periods:
M-LIGHT reading (a) takes the epochs and touches inside each light trial's own light period, and
inside the median light period of the neuron's light trials for control trials; reading (b) takes
whole trials.

In [ ]:
def site_of(c, key):
    return c["meta"]["opto_sites"].get(c["meta"]["lasers"].get(key, ""), ("", ""))


def light_info(c):
    """Per trial: dict of series on (power), first onset and last offset (s, whisker clock)."""
    n = c["meta"]["n_trials"]
    on = {k: np.array([np.nanmax(c["laser"][k][i]) for i in range(n)]) for k in (c["laser"] or {})}
    return on


def light_trials(c, kind):
    if not c["laser"]:
        return np.zeros(c["meta"]["n_trials"], bool)
    P = light_info(c)
    keys = list(P)
    lit = {k: P[k] > 0.05 for k in keys}
    any_on = np.any([lit[k] for k in keys], 0)
    out = np.zeros(c["meta"]["n_trials"], bool)
    gd = c["meta"]["genotype"] + " " + c["meta"]["description"]
    vgat = "Slc32a1" in gd or "VGAT" in gd.upper()
    for k in keys:
        lam, loc = site_of(c, k)
        others = np.any([lit[j] for j in keys if j != k], 0) if len(keys) > 1 else np.zeros_like(any_on)
        if kind == "vpm" and lam == "473" and loc.startswith("VPM,"):
            out |= lit[k] & ~others
        if kind == "enphr" and lam == "594":
            out |= lit[k]
        if kind == "cortex" and lam == "473" and loc.startswith("vS1") and vgat:
            out |= lit[k] & ~others
    return out & c["good"]


def light_window(c, i, kind):
    w = []
    for k in c["laser_on"] or {}:
        a, b = c["laser_on"][k][i]
        if np.isfinite(a):
            w.append((a, b))
    if not w:
        return None
    return min(x[0] for x in w), max(x[1] for x in w)


def light_power(c, i):
    return max([np.nanmax(c["laser"][k][i]) for k in c["laser"]] or [0.0])


def windows(c, trials, kind, ctl_window):
    """Per-trial windows (ms): the trial's own light period, or `ctl_window` for trials without light."""
    out = {}
    for i in np.flatnonzero(trials):
        w = light_window(c, i, kind)
        out[i] = (w[0] * 1000, w[1] * 1000) if w is not None else ctl_window
    return out


def light_summary(c, kind, reading="b", margin="a", min_iti=0.025, spt_kind="cortex5", top_power=False):
    """Rates in epochs and spikes per touch in control and light trials (M-LIGHT (a) or (b));
    with `top_power`, only the light trials at the neuron's highest power (M-LIGHT reading d)."""
    out = {}
    ctl = ssc7.control_trials(c)
    las = light_trials(c, kind)
    if top_power and las.any():
        pw = np.round([light_power(c, i) for i in range(len(las))], 1)
        las = las & (pw == pw[las].max())
    spk = ssc7.spikes_by_trial(c)
    lw = [light_window(c, i, kind) for i in np.flatnonzero(las)]
    med = (np.median([w[0] for w in lw]) * 1000, np.median([w[1] for w in lw]) * 1000) if lw else None
    for lab, tr in (("ctl", ctl), ("las", las)):
        ep = ssc7.cell_epochs(c, tr, margin)
        tt = ssc7.touch_table(c, min_iti, tr)
        if reading == "a" and med is not None:
            W = windows(c, tr, kind, med)
            for i, d in ep.items():
                a0, b0 = W[i]
                d["whisk"] = [(a, b) for a, b in d["whisk"] if a >= a0 and b <= b0]
                d["nonw"] = [(max(a, a0), min(b, b0)) for a, b in d["nonw"] if min(b, b0) - max(a, a0) >= 100]
            if len(tt):
                keep = [W[int(i)][0] <= t * 1000 < W[int(i)][1] for i, t in tt]
                tt = tt[np.array(keep, bool)]
        w = ssc7.epoch_rates(spk, ep, "whisk")
        n = ssc7.epoch_rates(spk, ep, "nonw")
        out[lab] = dict(n_trials=int(tr.sum()), whisk=ssc7.rate_readings(w)[0], nonw=ssc7.rate_readings(n)[0],
                        whisk_s=w[:, 2].sum() if len(w) else 0, nonw_s=n[:, 2].sum() if len(n) else 0,
                        spt=ssc7.spikes_per_touch(spk, tt, spt_kind) if len(tt) else np.nan, n_touch=len(tt),
                        ep=ep, tt=tt, w_rows=w, n_rows=n)
    return out


LIGHT = {}
for c in cells:
    n_ = ssc7.cell_name(c)
    if GROUP[n_] == "VPM":
        continue
    LIGHT[n_] = {k: int(light_trials(c, k).sum()) for k in ("vpm", "enphr", "cortex")}
LT = pd.DataFrame(LIGHT).T
LT["group"] = [GROUP[n] for n in LT.index]
print("neurons with light trials of each experiment, by group (neurons, trials):")
print(LT.groupby("group").agg(vpm_neurons=("vpm", lambda x: int((x > 0).sum())), vpm_trials=("vpm", "sum"),
                              enphr_neurons=("enphr", lambda x: int((x > 0).sum())), enphr_trials=("enphr", "sum"),
                              cortex_neurons=("cortex", lambda x: int((x > 0).sum())), cortex_trials=("cortex", "sum")).to_string())
pw_ = pd.Series({n: tuple(sorted({round(light_power(NAME[n], i), 1) for i in np.flatnonzero(light_trials(NAME[n], "vpm"))})) for n in LT.index[LT.vpm > 0]})
print("VPM light powers (mW) per neuron:", pw_.value_counts().to_dict())

## thalamus-inhibition-vpm

Blocked, data missing: the 17 thalamic multiunit recordings with patch pipettes (Fig. 6b,c) are
not in the data set (the data check lists every folder and file; the only thalamic recordings
are the 29 silicon-probe units without photostimulation).

## thalamus-inhibition-fs

> Thalamic photoinhibition reduces L4 FS spike rates during nonwhisking (to 4% of control, n =
> 13), whisking (to 13%, n = 15) and touch (to 32%, n = 14).

The FS neurons with thalamic photoinhibition trials. A neuron enters an epoch's comparison when
it has at least 1 s of that epoch and (for touch) at least 5 touches in both trial types; the
differing n of the paper are read this way (criterion).

In [ ]:
def light_table(group_names, kind, reading, power=None):
    rows = []
    for n in group_names:
        c = NAME[n]
        if light_trials(c, kind).sum() == 0:
            continue
        L = light_summary(c, kind, reading, top_power=power == "top")
        rows.append(dict(name=n, light_trials=L["las"]["n_trials"], control_trials=L["ctl"]["n_trials"],
                         nonw_ctl=L["ctl"]["nonw"] if L["ctl"]["nonw_s"] >= 1 else np.nan,
                         nonw_las=L["las"]["nonw"] if L["las"]["nonw_s"] >= 1 else np.nan,
                         whisk_ctl=L["ctl"]["whisk"] if L["ctl"]["whisk_s"] >= 1 else np.nan,
                         whisk_las=L["las"]["whisk"] if L["las"]["whisk_s"] >= 1 else np.nan,
                         spt_ctl=L["ctl"]["spt"] if L["ctl"]["n_touch"] >= 5 else np.nan,
                         spt_las=L["las"]["spt"] if L["las"]["n_touch"] >= 5 else np.nan))
    return pd.DataFrame(rows)


def pct_report(T, cols, label):
    res = {}
    for m, (a, b) in cols.items():
        ok = T[a].notna() & T[b].notna()
        x, y = T.loc[ok, a].values, T.loc[ok, b].values
        if ok.sum() < 2:
            print(f"{label} {m}: n = {ok.sum()}")
            continue
        p = stats.wilcoxon(y, x).pvalue
        e = 100 * y.mean() / x.mean()
        r = y[x > 0] / x[x > 0]
        f = 100 * np.median(r) if len(r) else np.nan
        res[m] = (e, f, p, int(ok.sum()), y.mean() < x.mean())
        print(f"{label} {m}: n = {ok.sum()}, control {x.mean():.2f}, light {y.mean():.2f}; (e) ratio of means {e:.0f}%, (f) median ratio {f:.0f}%; signed-rank p = {p:.2g}")
    return res


TH = {}
for rd in "ab":
    T = light_table(FS, "vpm", rd)
    print(f"--- M-LIGHT reading ({rd}): FS neurons with thalamic photoinhibition trials: {len(T)}")
    TH[rd] = pct_report(T, {"nonwhisking": ("nonw_ctl", "nonw_las"), "whisking": ("whisk_ctl", "whisk_las"), "touch": ("spt_ctl", "spt_las")}, "FS")
    if rd == "a":
        print(T.round(2).to_string(index=False))
# M-LIGHT reading (d): the highest light power of each neuron only, within the light period (a)
T = light_table(FS, "vpm", "a", power="top")
print("--- M-LIGHT readings (a) and (d): FS neurons, light trials at their highest power only:", len(T), "; light trials per neuron", T.light_trials.tolist())
TH["ad"] = pct_report(T, {"nonwhisking": ("nonw_ctl", "nonw_las"), "whisking": ("whisk_ctl", "whisk_las"), "touch": ("spt_ctl", "spt_las")}, "FS")
exp = {"nonwhisking": (4, 0.4), "whisking": (13, 1.3), "touch": (32, 3.2)}
for rd in ("a", "b", "ad"):
    for col in (0, 1):
        print(f"reading ({rd}), {'(e) ratio of means' if col == 0 else '(f) median ratio'}: within tolerance "
              f"{[within(TH[rd][m][col], *exp[m]) for m in exp if m in TH[rd]]}; tests p < 0.05 with light lower: {[bool(TH[rd][m][2] < 0.05 and TH[rd][m][4]) for m in exp if m in TH[rd]]}")

**Result: partially reproduced.** Within each trial's light period (M-LIGHT reading a), thalamic photoinhibition lowers FS rates to 12% of control in nonwhisking (ratio of means; median ratio 9%, n = 12, signed-rank p = 0.00098), 27% in whisking (24%, n = 14, p = 0.0017) and 47% for touch (47%, n = 14, p = 0.00037). All three tests hold. None of the percentages is within tolerance of 4%, 13% and 32%: the inhibition here is weaker. Over whole trials (reading b), the reductions are smaller still (85%, 67%, 69%), and the nonwhisking test fails (p = 0.71). With only each neuron's highest power (reading d, within the light period), the rates fall to 6%, 22% and 38% (ratio of means) or 2%, 13% and 35% (median ratios), all tests holding. Under (d) with median ratios the whisking and touch values are within tolerance, but the nonwhisking 2% is outside 4 +- 0.4, so no reading has all three.

## thalamus-inhibition-dose

> The reduction of L4 FS spiking by thalamic photoinhibition is dose dependent.

In [ ]:
def per_trial_epoch_rates(c, kind):
    """Per trial (power, nonwhisking rate, whisking rate, spikes per touch) inside the light window
    (reading a), over control and light trials."""
    ctl = ssc7.control_trials(c)
    las = light_trials(c, kind)
    tr = ctl | las
    lw = [light_window(c, i, kind) for i in np.flatnonzero(las)]
    med = (np.median([w[0] for w in lw]) * 1000, np.median([w[1] for w in lw]) * 1000)
    W = windows(c, tr, kind, med)
    spk = ssc7.spikes_by_trial(c)
    ep = ssc7.cell_epochs(c, tr)
    T_ = ssc7.touch_table(c, 0.025, tr)
    rows = []
    for i, d in ep.items():
        a0, b0 = W[i]
        s = spk[i]
        out = [light_power(c, i) if las[i] else 0.0]
        for k in ("nonw", "whisk"):
            segs = [(max(a, a0), min(b, b0)) for a, b in d[k] if min(b, b0) - max(a, a0) >= (100 if k == "nonw" else 1)] if k == "nonw" else [(a, b) for a, b in d[k] if a >= a0 and b <= b0]
            dur = sum(b - a for a, b in segs) / 1000
            cnt = sum(np.sum((s >= a / 1000) & (s < b / 1000)) for a, b in segs)
            out.append(cnt / dur if dur > 0 else np.nan)
        tt = T_[(T_[:, 0] == i) & (T_[:, 1] * 1000 >= a0) & (T_[:, 1] * 1000 < b0)] if len(T_) else T_
        out.append(ssc7.spikes_per_touch(spk, tt) if len(tt) else np.nan)
        rows.append(out)
    return np.array(rows, float)


rho = {"nonw": [], "whisk": [], "touch": []}
for n in FS:
    c = NAME[n]
    if light_trials(c, "vpm").sum() == 0:
        continue
    R = per_trial_epoch_rates(c, "vpm")
    for j, k in enumerate(("nonw", "whisk", "touch")):
        ok = np.isfinite(R[:, j + 1])
        if ok.sum() >= 10 and len(np.unique(R[ok, 0])) >= 2:
            rho[k].append(stats.spearmanr(R[ok, 0], R[ok, j + 1]).statistic)
neg = 0
for k, v in rho.items():
    v = np.array(v)
    v = v[np.isfinite(v)]
    p = stats.wilcoxon(v).pvalue if len(v) > 1 else np.nan
    neg += int(p < 0.05 and np.median(v) < 0)
    print(f"{k}: Spearman rho (power with control as 0, rate) over {len(v)} FS neurons: median {np.median(v):.2f}, negative in {int((v < 0).sum())}; signed-rank against 0 p = {p:.2g}")
print("epochs with a significant negative dose relation:", neg, "of 3")
levels = []
for n in FS:
    c = NAME[n]
    if light_trials(c, "vpm").sum():
        levels.append(len({round(light_power(c, i), 1) for i in np.flatnonzero(light_trials(c, "vpm"))}))
print("FS neurons with 1 or 2 light powers:", pd.Series(levels).value_counts().to_dict())
# variant: ratio of mean whisking rate at each power to control, pooled over neurons
pooled = []
for n in FS:
    c = NAME[n]
    if light_trials(c, "vpm").sum() == 0:
        continue
    R = per_trial_epoch_rates(c, "vpm")
    for pw in np.unique(R[:, 0]):
        sel = R[:, 0] == pw
        pooled.append(dict(name=n, power=pw, nonw=np.nanmean(R[sel, 1]), whisk=np.nanmean(R[sel, 2]), spt=np.nanmean(R[sel, 3])))
PP = pd.DataFrame(pooled)
print(PP.groupby("power")[["nonw", "whisk", "spt"]].agg(["mean", "count"]).round(2).to_string())
# variant: light trials only (control trials left out), for the neurons with two light powers, so
# that the correlation tests dose and not light against no light
rho2 = {"nonw": [], "whisk": [], "touch": []}
for n in FS:
    c = NAME[n]
    if light_trials(c, "vpm").sum() == 0:
        continue
    R = per_trial_epoch_rates(c, "vpm")
    R = R[R[:, 0] > 0]
    for j, k in enumerate(("nonw", "whisk", "touch")):
        ok = np.isfinite(R[:, j + 1])
        if ok.sum() >= 10 and len(np.unique(R[ok, 0])) >= 2:
            rho2[k].append(stats.spearmanr(R[ok, 0], R[ok, j + 1]).statistic)
for k, v in rho2.items():
    v = np.array(v)
    v = v[np.isfinite(v)]
    p = stats.wilcoxon(v).pvalue if len(v) > 1 else np.nan
    print(f"variant, light trials only, {k}: Spearman rho (power, rate) over {len(v)} FS neurons with two powers: median {np.median(v) if len(v) else np.nan:.2f}, negative in {int((v < 0).sum())}; signed-rank against 0 p = {p:.2g}")

**Result: reproduced.** Per FS neuron, the Spearman correlation between light power (control as 0) and the rate inside the light period is negative in 12 of 13 neurons for nonwhisking (median rho -0.47, signed-rank p = 0.00049), 13 of 14 for whisking (-0.65, p = 0.0012) and 12 of 14 for touch (-0.36, p = 0.0012). Pooled over neurons, the mean whisking rate falls with power: 21.97, 9.32 and 5.14 spikes/s at 0, 3.75 and 15 mW. Ten of the 14 neurons have two light powers, four have one. The test as fixed mostly compares light with no light. Among light trials only, in the neurons with two powers, the correlation is weaker: median rho -0.58 (nonwhisking, 6 neurons, p = 0.31), -0.28 (whisking, 10, p = 0.074) and -0.23 (touch, 7, p = 0.031), so only one epoch shows a dose relation beyond light against no light.

## thalamus-inhibition-l4e-vm

> Thalamic photoinhibition hyperpolarizes L4 excitatory neurons and reduces their Vm
> fluctuations, with an onset latency of 4.2 ms, longer than the 2.8 ms of direct cortical
> GABAergic activation.

In [ ]:
from scipy import signal as sps

HP = sps.butter(2, 10, btype="highpass", fs=ssc7.VM_FS)


def vm_outside_touch(c, P, tr, win=None):
    T_ = ssc7.touches(c)
    vs, vv = [], []
    for i in np.flatnonzero(tr):
        if i >= len(P["vm"]):
            continue
        v = P["vm"][i]
        m = np.zeros(len(v), bool)
        a0, b0 = (0.0, 5.0) if win is None else win(i)
        m[max(0, ssc7.vm_at(c, P["vm"], i, a0)):ssc7.vm_at(c, P["vm"], i, b0)] = True
        for x in T_[i][0]:
            kk = ssc7.vm_at(c, P["vm"], i, x)
            m[max(0, kk - 100):kk + 1000] = False
        vs.append(v[m])
        vv.append(sps.filtfilt(*HP, v)[m])
    if not vs:
        return np.nan, np.nan
    return float(np.median(np.concatenate(vs))), float(np.var(np.concatenate(vv)))


def onset_average(c, P, tr, pre=0.05, post=0.1):
    X = []
    for i in np.flatnonzero(tr):
        w = light_window(c, i, None)
        if w is None or i >= len(P["vm"]):
            continue
        k = ssc7.vm_at(c, P["vm"], i, w[0])
        v = P["vm"][i]
        a, b = k - int(pre * ssc7.VM_FS), k + int(post * ssc7.VM_FS)
        if a >= 0 and b <= len(v):
            X.append(v[a:b])
    return np.mean(X, 0) if X else None


def neg_latency(avg, pre=0.05, fs=ssc7.VM_FS):
    """M-LAT Vm rule for a hyperpolarization: first time dVm/dt falls below the mean minus 3 s.d.
    of dVm/dt over the 25 ms before light onset."""
    z = int(pre * fs)
    d = np.gradient(avg) * fs
    base = d[z - int(0.025 * fs):z]
    k = np.flatnonzero(d[z:z + int(0.05 * fs)] < base.mean() - 3 * base.std())
    return np.nan if len(k) == 0 else 1000.0 * k[0] / fs


def neg_latency_smooth(avg, pre=0.05, fs=ssc7.VM_FS, smooth=10, hold=10):
    """Variant of the rule above, robust to the light artifact at onset (an assumption): dVm/dt of
    Vm smoothed over 1 ms, baseline 25 ms before onset, and the crossing must last 1 ms."""
    z = int(pre * fs)
    v = np.convolve(avg, np.ones(smooth) / smooth, mode="same")
    d = np.gradient(v) * fs
    base = d[z - int(0.025 * fs):z - smooth]
    below = d[z:z + int(0.05 * fs)] < base.mean() - 3 * base.std()
    for k in range(len(below) - hold):
        if below[k:k + hold].all():
            return 1000.0 * k / fs
    return np.nan


rows, avgs = [], {"vpm": [], "cortex": []}
for n in L4EWC:
    c = NAME[n]
    P = ssc7.prep_vm(c)
    for kind in ("vpm", "cortex"):
        las = light_trials(c, kind)
        if las.sum() == 0:
            continue
        av = onset_average(c, P, las)
        if av is not None:
            avgs[kind].append(av)
        if kind == "vpm":
            lw = [light_window(c, i, kind) for i in np.flatnonzero(las)]
            med = (np.median([w[0] for w in lw]), np.median([w[1] for w in lw]))
            win = lambda i, c=c, med=med: light_window(c, i, "vpm") or med
            vc, varc = vm_outside_touch(c, P, ssc7.control_trials(c), win)
            vl, varl = vm_outside_touch(c, P, las, win)
            vc_b, varc_b = vm_outside_touch(c, P, ssc7.control_trials(c))
            vl_b, varl_b = vm_outside_touch(c, P, las)
            rows.append(dict(name=n, n_light=int(las.sum()), vm_ctl=vc, vm_las=vl, var_ctl=varc, var_las=varl,
                             vm_ctl_b=vc_b, vm_las_b=vl_b, var_ctl_b=varc_b, var_las_b=varl_b,
                             latency_own=neg_latency(av) if av is not None else np.nan))
TL = pd.DataFrame(rows)
print(TL.round(3).to_string(index=False))
for rd, sfx in (("a", ""), ("b", "_b")):
    p1 = stats.wilcoxon(TL["vm_las" + sfx], TL["vm_ctl" + sfx]).pvalue
    p2 = stats.wilcoxon(TL["var_las" + sfx], TL["var_ctl" + sfx]).pvalue
    print(f"M-LIGHT ({rd}), n = {len(TL)}: Vm outside touch, light minus control median {np.median(TL['vm_las' + sfx] - TL['vm_ctl' + sfx]):.2f} mV, lower in {int((TL['vm_las' + sfx] < TL['vm_ctl' + sfx]).sum())}, p = {p1:.2g}; "
          f"10 Hz high-passed variance ratio median {np.median(TL['var_las' + sfx] / TL['var_ctl' + sfx]):.2f}, lower in {int((TL['var_las' + sfx] < TL['var_ctl' + sfx]).sum())}, p = {p2:.2g}")
for rule, f in (("M-LAT Vm rule", neg_latency), ("variant, 1 ms smoothing and 1 ms hold", neg_latency_smooth)):
    lat = {}
    for kind in ("vpm", "cortex"):
        if avgs[kind]:
            g = np.mean(avgs[kind], 0)
            lat[kind] = f(g)
            z = int(0.05 * ssc7.VM_FS)
            d0 = np.gradient(g)[z] * ssc7.VM_FS
            print(f"{rule}, {kind}: grand average over {len(avgs[kind])} neurons; Vm change 5-50 ms after onset {g[z + 50:z + 500].mean() - g[z - 250:z].mean():.2f} mV; "
                  f"dVm/dt at the onset sample {d0:.0f} mV/s; latency {lat[kind]:.1f} ms; per-neuron median {np.nanmedian([f(a) for a in avgs[kind]]):.1f} ms")
    print(f"   thalamic within 4.2+-0.42: {within(lat.get('vpm', np.nan), 4.2, 0.42)}; cortical within 2.8+-0.28: {within(lat.get('cortex', np.nan), 2.8, 0.28)}; thalamic longer: {lat.get('vpm', np.nan) > lat.get('cortex', np.nan)}")
# variant: the smoothing and hold windows of the onset-latency rule
grid = []
for sm in (5, 10, 20):
    for hd in (5, 10, 20):
        lv = neg_latency_smooth(np.mean(avgs["vpm"], 0), smooth=sm, hold=hd)
        lc = neg_latency_smooth(np.mean(avgs["cortex"], 0), smooth=sm, hold=hd)
        grid.append(dict(smooth_ms=sm / 10, hold_ms=hd / 10, thalamic=lv, cortical=lc, thalamic_longer=lv > lc,
                         thalamic_within=within(lv, 4.2, 0.42), cortical_within=within(lc, 2.8, 0.28)))
print("onset latency (ms) on the grand averages, by smoothing and hold window:")
print(pd.DataFrame(grid).round(2).to_string(index=False))

**Result: partially reproduced.** Within the light period, Vm outside touch is lower with thalamic photoinhibition in all 13 L4 E neurons (median -2.99 mV, signed-rank p = 0.00024). The 10 Hz high-passed variance falls to a median of 0.59 of control (lower in 13, p = 0.00024). Both tests hold. The latency rule as the claims file gives it crosses at the onset sample itself, giving 0.0 ms for both grand averages; at that sample dVm/dt is -295 and -392 mV/s, a step the light makes. With 1 ms smoothing and a 1 ms hold (assumed), the latency is 4.8 ms for thalamic photoinhibition (13 neurons) and 2.6 ms for direct cortical activation (8 neurons). The thalamic latency is the longer, as the paper says. The cortical one is within 2.8 +- 0.28; the thalamic one is outside 4.2 +- 0.42. Over smoothing windows of 0.5 to 2 ms and holds of 0.5 to 2 ms, the thalamic latency is 4.5 to 5.0 ms and the cortical 2.0 to 2.7 ms: the thalamic one is always longer, and no window puts both within tolerance (2 ms smoothing gives 4.5 and 2.0 ms, thalamic within and cortical outside).

## enphr-kruskal-wallis

> Partially suppressing PV (FS) interneurons with eNpHR changes the responses of L4 excitatory
> neurons (P = 1e-6, Kruskal-Wallis).

The neurons are the L4 E neurons (whole-cell and cell-attached) with 594 nm light trials.

In [ ]:
EN = {}
for rd in "ab":
    rows = []
    for n in L4E85:
        c = NAME[n]
        if light_trials(c, "enphr").sum() == 0:
            continue
        L = light_summary(c, "enphr", rd)
        L8 = light_summary(c, "enphr", rd, min_iti=0.010, spt_kind="cortex8")
        rows.append(dict(name=n, group=GROUP[n], mouse=c["meta"]["subject"], light_trials=L["las"]["n_trials"], control_trials=L["ctl"]["n_trials"],
                         nonw_ctl=L["ctl"]["nonw"], nonw_las=L["las"]["nonw"], whisk_ctl=L["ctl"]["whisk"], whisk_las=L["las"]["whisk"],
                         spt8_ctl=L8["ctl"]["spt"], spt8_las=L8["las"]["spt"]))
    EN[rd] = pd.DataFrame(rows)
E = EN["a"]
print(f"L4E neurons with eNpHR trials: {len(E)} ({E.group.value_counts().to_dict()}) in {E.mouse.nunique()} mice; paper n = 40")
for rd, D in EN.items():
    D = D.dropna(subset=["nonw_ctl", "nonw_las", "whisk_ctl", "whisk_las"])
    h4 = stats.kruskal(D.nonw_ctl, D.whisk_ctl, D.nonw_las, D.whisk_las)
    h2 = stats.kruskal(np.r_[D.nonw_ctl, D.whisk_ctl], np.r_[D.nonw_las, D.whisk_las])
    print(f"M-LIGHT ({rd}), n = {len(D)}: Kruskal-Wallis (a) four groups H = {h4.statistic:.1f}, p = {h4.pvalue:.2g}; (b) two groups H = {h2.statistic:.1f}, p = {h2.pvalue:.2g}")
# The eNpHR neurons against the cell list: the cell list's L4 E rows from PV-IRES-Cre mice
cl_e = CL[CL.section.isin(["Whole-cell units", "Extracellular units"])]
cl_pv = cl_e[cl_e.mouse_type.astype(str).str.lower().str.contains("pv-ires-cre")]
e_keys = {key(n[:6] if n.startswith("JY") else n) for n in E.name}
pv_keys = {key(x) for x in cl_pv.cell}
print(f"cell list L4 E rows from PV-IRES-Cre mice: {len(cl_pv)} ({cl_pv.section.value_counts().to_dict()}), in {cl_pv.mouse.nunique()} mice")
print("  eNpHR neurons here not among them:", sorted(n for n in E.name if key(n[:6] if n.startswith('JY') else n) not in pv_keys))
print("  of them, not here:", sorted(x for x in cl_pv.cell if key(x) not in e_keys))

**Result: reproduced.** Over the 43 L4 E neurons with eNpHR trials, within the light period (M-LIGHT reading a), the Kruskal-Wallis test of the four groups (nonwhisking and whisking, control and eNpHR) gives H = 18.0, p = 0.00044. The two-group reading gives p = 0.00022. Over whole trials, the four-group test gives p = 0.021. The paper's P is 1e-6. The cell list names 43 L4 E neurons from PV-IRES-Cre mice (38 extracellular, 5 whole-cell). The 43 here are those, less JY0781 (no eNpHR light in its file) and plus JY1268, which the cell list does not name. The paper's 40 cannot be matched to a subset.

## enphr-l4e-rates

> Suppressing PV interneurons increases L4 excitatory spike rates during nonwhisking (0.6 +- 0.1
> to 1.6 +- 0.3 spikes/s, P = 6e-5) and whisking (0.8 +- 0.2 to 2.7 +- 0.5 spikes/s, P = 2e-6),
> mean +- s.e.m., n = 40.

In [ ]:
sem = lambda x: np.std(x, ddof=1) / np.sqrt(len(x))
for rd, D in EN.items():
    D = D.dropna(subset=["nonw_ctl", "nonw_las", "whisk_ctl", "whisk_las"])
    vals = [D.nonw_ctl.mean(), D.nonw_las.mean(), D.whisk_ctl.mean(), D.whisk_las.mean()]
    exp = [(0.6, 0.1), (1.6, 0.3), (0.8, 0.2), (2.7, 0.5)]
    p1 = stats.wilcoxon(D.nonw_las, D.nonw_ctl).pvalue
    p2 = stats.wilcoxon(D.whisk_las, D.whisk_ctl).pvalue
    print(f"M-LIGHT ({rd}), n = {len(D)}: nonwhisking {D.nonw_ctl.mean():.2f} +- {sem(D.nonw_ctl):.2f} -> {D.nonw_las.mean():.2f} +- {sem(D.nonw_las):.2f} (p = {p1:.2g}); "
          f"whisking {D.whisk_ctl.mean():.2f} +- {sem(D.whisk_ctl):.2f} -> {D.whisk_las.mean():.2f} +- {sem(D.whisk_las):.2f} (p = {p2:.2g}); "
          f"within: {[within(v_, e, t) for v_, (e, t) in zip(vals, exp)]}")
    if rd == "a":
        print("by mouse, light minus control whisking rate (mean):", D.assign(d=D.whisk_las - D.whisk_ctl).groupby("mouse").d.agg(["mean", "count"]).round(2).to_dict("index"))

**Result: partially reproduced.** Within the light period (n = 43), PV suppression raises the nonwhisking rate from 0.54 +- 0.11 to 1.45 +- 0.31 spikes/s (p = 0.00012) and the whisking rate from 1.13 +- 0.24 to 2.82 +- 0.54 (p = 3e-5). Both tests hold. Three of the four means are within tolerance; the control whisking mean, 1.13, is above 0.8 +- 0.2. Over whole trials the rises are smaller (0.59 to 0.78, 1.04 to 1.58).

## enphr-whisking-driven

> With PV interneurons suppressed, L4 excitatory neurons become driven by whisker movement
> (whisking against nonwhisking, P = 0.02).

In [ ]:
for rd, D in EN.items():
    D = D.dropna(subset=["nonw_ctl", "nonw_las", "whisk_ctl", "whisk_las"])
    p_l = stats.wilcoxon(D.whisk_las, D.nonw_las).pvalue
    p_c = stats.wilcoxon(D.whisk_ctl, D.nonw_ctl).pvalue
    print(f"M-LIGHT ({rd}), n = {len(D)}: eNpHR trials whisking {D.whisk_las.mean():.2f} vs nonwhisking {D.nonw_las.mean():.2f}, p = {p_l:.2g}; "
          f"control trials {D.whisk_ctl.mean():.2f} vs {D.nonw_ctl.mean():.2f}, p = {p_c:.2g}")

**Result: reproduced.** In eNpHR trials, whisking rates exceed nonwhisking rates: 2.82 against 1.45 spikes/s within the light period (signed-rank p = 0.013, n = 43), or 1.58 against 0.78 over whole trials (p = 0.0065). The same test in control trials gives p = 0.052 within the light period and p = 0.035 over whole trials.

## enphr-spikes-per-touch

> PV interneuron suppression changes touch-evoked spiking of L4 excitatory neurons
> heterogeneously, with no significant mean change (0.27 +- 0.06 to 0.32 +- 0.06 spikes per
> touch, n = 40, P = 0.09, Wilcoxon signed-rank); weakly responding neurons tend to respond
> more, and strongly driven neurons stay the same or decrease.

In [ ]:
for rd, D in EN.items():
    D = D.dropna(subset=["spt8_ctl", "spt8_las"])
    p = stats.wilcoxon(D.spt8_las, D.spt8_ctl).pvalue
    rho = stats.spearmanr(D.spt8_ctl, D.spt8_las - D.spt8_ctl)
    print(f"M-LIGHT ({rd}), n = {len(D)}: spikes per touch (4-20 ms minus 0-16 ms before, ITI >= 10 ms) {D.spt8_ctl.mean():.3f} +- {sem(D.spt8_ctl):.3f} -> "
          f"{D.spt8_las.mean():.3f} +- {sem(D.spt8_las):.3f}; signed-rank p = {p:.2g}; Spearman (control, change) rho = {rho.statistic:.2f}, p = {rho.pvalue:.2g}; "
          f"within: {within(D.spt8_ctl.mean(), 0.27, 0.06)}, {within(D.spt8_las.mean(), 0.32, 0.06)}")

**Result: not reproduced.** Both means are within tolerance. Within the light period, spikes per touch are 0.254 +- 0.063 in control and 0.342 +- 0.057 with eNpHR (0.27 +- 0.06, 0.32 +- 0.06). The claim of no mean change fails: signed-rank p = 0.009 (0.027 over whole trials). The negative correlation between control response and change is not significant (rho = -0.13, p = 0.42; over whole trials -0.22, p = 0.16). One of the three parts holds under either reading.

## enphr-auc

> PV interneuron suppression improves touch detection (AUC) for neurons with weak touch responses
> and diminishes it for neurons with robust touch responses.

In [ ]:
def auc(pos, neg):
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    r = stats.rankdata(np.r_[pos, neg])
    return (r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


def roc_cell(c, tr, reading, rng, skip_passive=True):
    spk = ssc7.spikes_by_trial(c)
    T_ = ssc7.touches(c)
    pin = np.asarray(c["pole_in"]) - c["t0"] if c["pole_in"] is not None else np.full(len(spk), 0.5)
    pout = np.asarray(c["pole_out"]) - c["t0"] if c["pole_out"] is not None else np.full(len(spk), 4.5)
    tags = c.get("epoch_tag") or []
    pos, ntr, ft = [], [], []
    for i in np.flatnonzero(tr):
        a = T_[i][0]
        s = spk[i]
        if skip_passive and i < len(tags) and "passive" in str(tags[i]):
            # the first touch of a trial tagged passive is left out, as in touch_table; a trial
            # whose only touch is that one is neither a touch nor a no-touch trial
            if len(a) <= 1:
                continue
            a = a[1:]
        if len(a):
            t = a[0]
            ft.append(t)
            pos.append(np.sum((s >= t) & (s < t + 0.25)) - np.sum((s >= t - 0.25) & (s < t)))
        else:
            ntr.append(i)
    if len(pos) < 3 or len(ntr) < 3:
        return np.nan
    if reading == "a":
        neg = []
        for _ in range(5000):
            i = ntr[rng.integers(len(ntr))]
            lo, hi = max(0.25, pin[i]), min(4.75, pout[i] if pout[i] > pin[i] else 4.75)
            t = rng.uniform(lo, max(lo + 1e-3, hi))
            s = spk[i]
            neg.append(np.sum((s >= t) & (s < t + 0.25)) - np.sum((s >= t - 0.25) & (s < t)))
        return auc(pos, neg)
    t = np.median(ft)
    neg = np.array([np.sum((spk[i] >= t) & (spk[i] < t + 0.25)) - np.sum((spk[i] >= t - 0.25) & (spk[i] < t)) for i in ntr])
    pos = np.array(pos)
    return np.mean([auc(rng.choice(pos, len(pos)), rng.choice(neg, len(neg))) for _ in range(5000)])


def touch_rates(c, tr):
    spk = ssc7.spikes_by_trial(c)
    tt = ssc7.touch_table(c, 0.025, tr)
    if len(tt) == 0:
        return np.nan, np.nan
    post = np.mean([np.sum((spk[int(i)] - t >= 0) & (spk[int(i)] - t < 0.25)) for i, t in tt]) / 0.25
    pre = np.mean([np.sum((spk[int(i)] - t >= -0.25) & (spk[int(i)] - t < 0)) for i, t in tt]) / 0.25
    return post, pre


rng = np.random.default_rng(4)
rows = []
for n in E.name:
    c = NAME[n]
    ctl, las = ssc7.control_trials(c), light_trials(c, "enphr")
    r = dict(name=n)
    for rd in "ab":
        r[f"auc_ctl_{rd}"] = roc_cell(c, ctl, rd, rng)
        r[f"auc_las_{rd}"] = roc_cell(c, las, rd, rng)
    r["touch_ctl"], r["base_ctl"] = touch_rates(c, ctl)
    r["touch_las"], r["base_las"] = touch_rates(c, las)
    rows.append(r)
AUC = pd.DataFrame(rows)
print("neurons without enough touch and no-touch trials for an AUC:", int(AUC[["auc_ctl_a", "auc_las_a"]].isna().any(axis=1).sum()), "of", len(AUC))
for rd in "ab":
    D = AUC.dropna(subset=[f"auc_ctl_{rd}", f"auc_las_{rd}", "touch_ctl"])
    dA = D[f"auc_las_{rd}"] - D[f"auc_ctl_{rd}"]
    ev = D.touch_ctl - D.base_ctl
    rho = stats.spearmanr(ev, dA)
    up, dn = dA > 0, dA < 0
    pv = stats.ranksums(ev[dn], ev[up]).pvalue if up.any() and dn.any() else np.nan
    print(f"M-ROC ({rd}), n = {len(D)}: AUC control median {D[f'auc_ctl_{rd}'].median():.3f}, eNpHR {D[f'auc_las_{rd}'].median():.3f}; "
          f"Spearman (control touch-evoked rate, AUC change) rho = {rho.statistic:.2f}, p = {rho.pvalue:.2g}; AUC up {int(up.sum())}, down {int(dn.sum())}; "
          f"variant rank-sum of evoked rate, down vs up: medians {ev[dn].median():.2f} vs {ev[up].median():.2f}, p = {pv:.2g}")
# Regression to the mean: the control touch-evoked rate and the control AUC come from the same
# trials, so their noise is shared and the correlation with (eNpHR - control) AUC is biased
# negative. Control trials are split at random into halves: the evoked rate from one half, the
# control AUC from the other (reading b). (i) estimates the bias with no light at all, the AUC
# difference between the halves; (ii) is the claim's test without the shared trials.
rng = np.random.default_rng(14)
rows = []
for n in E.name:
    c = NAME[n]
    ctl, las = np.flatnonzero(ssc7.control_trials(c)), light_trials(c, "enphr")
    perm = rng.permutation(ctl)
    h1, h2 = np.zeros(len(las), bool), np.zeros(len(las), bool)
    h1[perm[: len(perm) // 2]] = True
    h2[perm[len(perm) // 2:]] = True
    t1, b1 = touch_rates(c, h1)
    rows.append(dict(name=n, ev1=t1 - b1, auc1=roc_cell(c, h1, "b", rng), auc2=roc_cell(c, h2, "b", rng), auc_las=roc_cell(c, las, "b", rng)))
SH = pd.DataFrame(rows).dropna()
r1 = stats.spearmanr(SH.ev1, SH.auc2 - SH.auc1)
r2 = stats.spearmanr(SH.ev1, SH.auc_las - SH.auc2)
print(f"split control trials, n = {len(SH)}: (i) no light, evoked rate (half 1) against AUC half 2 minus half 1: rho = {r1.statistic:.2f}, p = {r1.pvalue:.2g}; "
      f"(ii) evoked rate (half 1) against eNpHR AUC minus control AUC (half 2): rho = {r2.statistic:.2f}, p = {r2.pvalue:.2g}")
# variant: the first touch of trials tagged passive kept as the trial's first touch (the rule of
# the first run of this notebook), reading (b)
rng = np.random.default_rng(15)
PV_ = pd.DataFrame([dict(name=n, a_ctl=roc_cell(NAME[n], ssc7.control_trials(NAME[n]), "b", rng, skip_passive=False),
                         a_las=roc_cell(NAME[n], light_trials(NAME[n], "enphr"), "b", rng, skip_passive=False)) for n in E.name]).merge(AUC[["name", "touch_ctl", "base_ctl"]])
PV_ = PV_.dropna()
r3 = stats.spearmanr(PV_.touch_ctl - PV_.base_ctl, PV_.a_las - PV_.a_ctl)
n_pass = sum(int(sum("passive" in str(t) for t in (NAME[n].get("epoch_tag") or []))) for n in E.name)
print(f"variant, passive first touches kept, M-ROC (b), n = {len(PV_)}: rho = {r3.statistic:.2f}, p = {r3.pvalue:.2g}; trials tagged passive in these neurons: {n_pass}")

**Result: not reproduced.** With the first touch of trials tagged passive left out, as everywhere else in this notebook (an assumption), the AUC change correlates negatively with the control touch-evoked rate, but not significantly: Spearman rho = -0.23, p = 0.16 under M-ROC reading (a) and rho = -0.28, p = 0.09 under reading (b) (n = 37). The rank-sum variant is not significant either (p = 0.1 and 0.43). Keeping those passive first touches, as the first run of this notebook did, gives rho = -0.34, p = 0.038 under reading (b) (143 trials tagged passive in these neurons), so the outcome rests on that choice. Regression to the mean does not drive the correlation here: with control trials split in halves, the evoked rate of one half does not correlate with the AUC difference between the halves (rho = 0.02, p = 0.91). With the evoked rate and the control AUC taken from different halves, the claim's correlation is rho = -0.19, p = 0.27.

## enphr-auc-groups

> Of the 40 neurons, eNpHR stimulation increased AUC in 19 and reduced it in 21. Those with
> increased AUC raised their touch-evoked rates (3.4 +- 0.8 to 8.7 +- 1.9 spikes/s, P = 0.0001);
> those with reduced AUC kept their touch-evoked rates (7.3 +- 1.5 to 7.0 +- 1.4 spikes/s, P =
> 0.98) but raised their baseline rates (0.9 +- 0.2 to 2.6 +- 0.6 spikes/s, P = 0.0002).

In [ ]:
exp = [(3.4, 0.8), (8.7, 1.9), (7.3, 1.5), (7.0, 1.4), (0.9, 0.2), (2.6, 0.6)]
for rd in "ab":
    D = AUC.dropna(subset=[f"auc_ctl_{rd}", f"auc_las_{rd}", "touch_ctl", "touch_las"]).copy()
    dA = D[f"auc_las_{rd}"] - D[f"auc_ctl_{rd}"]
    for ev in ("a", "b"):
        tc = D.touch_ctl if ev == "a" else D.touch_ctl - D.base_ctl
        tl = D.touch_las if ev == "a" else D.touch_las - D.base_las
        up, dn = (dA > 0).values, (dA < 0).values
        p_up = stats.wilcoxon(tl[up], tc[up]).pvalue
        p_dn = stats.wilcoxon(tl[dn], tc[dn]).pvalue
        p_bs = stats.wilcoxon(D.base_las[dn], D.base_ctl[dn]).pvalue
        vals = [tc[up].mean(), tl[up].mean(), tc[dn].mean(), tl[dn].mean(), D.base_ctl[dn].mean(), D.base_las[dn].mean()]
        print(f"M-ROC ({rd}), touch rate ({'raw' if ev == 'a' else 'baseline-subtracted'}): up {int(up.sum())}, down {int(dn.sum())}; "
              f"up group {vals[0]:.2f} -> {vals[1]:.2f} (p = {p_up:.2g}); down group {vals[2]:.2f} -> {vals[3]:.2f} (p = {p_dn:.2g}); "
              f"down-group baseline {vals[4]:.2f} -> {vals[5]:.2f} (p = {p_bs:.2g}); within: {[within(v_, e, t) for v_, (e, t) in zip(vals, exp)]}, "
              f"groups {within(up.sum(), 19, 1.9)}, {within(dn.sum(), 21, 2.1)}")

**Result: not reproduced.** The AUC rose in 17 and fell in 20 neurons (reading a), or 24 and 13 (reading b), against 19 and 21. Under no combination of readings are both group sizes and all six rates within tolerance; at most one rate is. With raw touch rates and reading (a), the increased-AUC group's touch-evoked rate rises from 2.04 to 3.16 spikes/s (p = 0.04), against 3.4 to 8.7, and the reduced-AUC group's baseline rise (3.53 to 4.20) is not significant (p = 0.083). Under reading (b) the increase is not significant (p = 0.069 to 0.084).

## enphr-pv-latency

> eNpHR stimulation strongly inhibits 3 of 4 recorded L4 PV interneurons; their spike rate falls
> 3 ms after laser onset and L4 excitatory activity rises only 2 ms later.

In [ ]:
def onset_psth(c, tr, pre=0.05, post=0.1):
    spk = ssc7.spikes_by_trial(c)
    H, n = np.zeros(int((pre + post) * 1000)), 0
    for i in np.flatnonzero(tr):
        w = light_window(c, i, None)
        if w is None:
            continue
        s = spk[i] - w[0]
        H += np.histogram(s[(s >= -pre) & (s < post)], bins=len(H), range=(-pre, post))[0]
        n += 1
    return H / max(n, 1) * 1000, n


def psth_latency(rate, sign, reading, pre=0.05):
    """Readings: (1) dPSTH/dt crosses 3 s.d. of its values over the 25 ms before onset (M-LAT adapted);
    (2) the rate crosses its baseline mean by 3 s.d. of the baseline rate. sign -1 for a decrease."""
    z = int(pre * 1000)
    if reading == 1:
        d = np.diff(rate)
        base = d[z - 25:z - 1]
        k = np.flatnonzero(sign * (d[z:z + 50] - base.mean()) > 3 * base.std())
        return np.nan if len(k) == 0 else float(k[0] + 1)
    base = rate[z - 25:z]
    k = np.flatnonzero(sign * (rate[z:z + 50] - base.mean()) > 3 * base.std())
    return np.nan if len(k) == 0 else float(k[0])


pv = [n for n in FS if light_trials(NAME[n], "enphr").sum()]
strong = 0
for n in pv:
    c = NAME[n]
    las, ctl = light_trials(c, "enphr"), ssc7.control_trials(c)
    lw = [light_window(c, i, None) for i in np.flatnonzero(las)]
    med = (np.median([w[0] for w in lw]), np.median([w[1] for w in lw]))
    spk = ssc7.spikes_by_trial(c)
    rl = [np.sum((spk[i] >= w[0]) & (spk[i] < w[1])) / (w[1] - w[0]) for i, w in zip(np.flatnonzero(las), lw)]
    rc = [np.sum((spk[i] >= med[0]) & (spk[i] < med[1])) / (med[1] - med[0]) for i in np.flatnonzero(ctl)]
    p = stats.ranksums(rl, rc).pvalue
    st_ = p < 0.05 and np.mean(rl) < 0.5 * np.mean(rc)
    strong += st_
    print(f"PV {n}: rate in the light window, control {np.mean(rc):.2f} (n={len(rc)}), light {np.mean(rl):.2f} (n={len(rl)}), ratio {np.mean(rl) / np.mean(rc):.2f}, rank-sum p = {p:.2g}; strongly inhibited: {st_}")
print("PV neurons strongly inhibited:", strong, "of", len(pv))
pv_rate = np.mean([onset_psth(NAME[n], light_trials(NAME[n], "enphr"))[0] for n in pv], 0)
e_rate = np.mean([onset_psth(NAME[n], light_trials(NAME[n], "enphr"))[0] for n in E.name], 0)
for rd in (1, 2):
    lp = psth_latency(pv_rate, -1, rd)
    le = psth_latency(e_rate, 1, rd)
    print(f"reading ({rd}): PV decrease latency {lp} ms, L4E increase latency {le} ms, delay {le - lp} ms; within (3+-1, 2+-1): {within(lp, 3, 1)}, {within(le - lp, 2, 1)}")
print("PV grand-average rate (spikes/s), -5 to +10 ms:", pv_rate[45:61].round(1).tolist())
print("L4E grand-average rate (spikes/s), -5 to +10 ms:", e_rate[45:61].round(2).tolist())

**Result: partially reproduced.** Three of the four PV neurons are strongly inhibited inside the light period: their rate falls to 0.05, 0.21 and 0.09 of control (rank-sum p < 1e-9). The fourth is unchanged (ratio 1.08, p = 0.64). Neither latency rule can read the latencies. The grand-average PV rate in 1 ms bins is sparse (several baseline bins at 0 spikes/s), so neither the dPSTH/dt rule nor the 3 s.d. rate rule finds the decrease. The L4 E increase crosses at 17 ms (dPSTH/dt) or 7 ms (rate). The 3 ms and the 2 ms delay are not reproduced.

## enphr-behavior

> With PV interneurons suppressed most mice (6 of 8) are more likely to report touch with the pole
> out of reach; in an example session the false alarm rate rose from 20% to 61%.

The data set holds the behavior of the recording sessions. The test is run on them: per mouse
with eNpHR trials, the proportion correct in Nogo trials with and without light, pooled over
the mouse's sessions, chi-squared test (Supplementary Fig. 14c legend).

In [ ]:
rows = []
for c in cells:
    n = ssc7.cell_name(c)
    if GROUP[n] == "VPM" or light_trials(c, "enphr").sum() == 0:
        continue
    tt = c["trial_type"]
    nogo = (tt[:, 2] + tt[:, 3]) > 0
    las = light_trials(c, "enphr")
    ctl = ssc7.control_trials(c)
    rows.append(dict(mouse=c["meta"]["subject"], session=n, cr_ctl=int((nogo & ctl & (tt[:, 2] > 0)).sum()), fa_ctl=int((nogo & ctl & (tt[:, 3] > 0)).sum()),
                     cr_las=int((nogo & las & (tt[:, 2] > 0)).sum()), fa_las=int((nogo & las & (tt[:, 3] > 0)).sum())))
B = pd.DataFrame(rows)
M = B.groupby("mouse")[["cr_ctl", "fa_ctl", "cr_las", "fa_las"]].sum()
out = []
for m, r in M.iterrows():
    tab = np.array([[r.cr_ctl, r.fa_ctl], [r.cr_las, r.fa_las]])
    p = stats.chi2_contingency(tab)[1] if (tab.sum(0) > 0).all() and (tab.sum(1) > 0).all() else np.nan
    pc, pl = r.cr_ctl / max(1, r.cr_ctl + r.fa_ctl), r.cr_las / max(1, r.cr_las + r.fa_las)
    out.append(dict(mouse=m, nogo_correct_ctl=round(pc, 2), nogo_correct_light=round(pl, 2), n_ctl=r.cr_ctl + r.fa_ctl, n_light=r.cr_las + r.fa_las, p=p,
                    decrease=bool(p < 0.05 and pl < pc), increase=bool(p < 0.05 and pl > pc)))
O = pd.DataFrame(out)
print(O.to_string(index=False))
print(f"mice: {len(O)}; significant decrease in Nogo performance with light: {int(O.decrease.sum())}; significant increase: {int(O.increase.sum())}")
B["fa_rate_ctl"] = B.fa_ctl / (B.fa_ctl + B.cr_ctl)
B["fa_rate_las"] = B.fa_las / (B.fa_las + B.cr_las)
B["dist"] = (B.fa_rate_ctl - 0.20).abs() + (B.fa_rate_las - 0.61).abs()
print("sessions closest to the example (FA 20% -> 61%):")
print(B.sort_values("dist").head(3)[["mouse", "session", "fa_rate_ctl", "fa_rate_las", "fa_ctl", "cr_ctl", "fa_las", "cr_las"]].round(2).to_string(index=False))
# The paper has 8 mice; which 8 of these 9 (if any) cannot be told. Dropping each mouse in turn:
drop1 = [int(O[O.mouse != m].decrease.sum()) for m in O.mouse]
print("significant decreases with one mouse left out:", dict(zip(O.mouse, drop1)), "; range", min(drop1), "to", max(drop1), "; within 6 +- 1:", any(within(x, 6, 1) for x in drop1))
print("light trials as a share of Nogo trials per mouse:", (O.n_light / (O.n_light + O.n_ctl)).round(2).tolist())

**Result: not reproduced.** In the recording sessions of the 9 mice with eNpHR trials, 4 show a significant decrease in Nogo performance with light (chi-squared p < 0.05) and none an increase, against 6 of 8 and 1. Leaving out any one mouse gives 3 or 4, so no 8 of these 9 mice match. No session matches the example's false alarm rates (20% to 61%); the closest goes from 14% to 58% (JY1247). The paper's behavior sessions may not be the recording sessions held here: the example is described as a behavior session with light on "a small subset of trials", while here light falls in 23% to 48% of Nogo trials.

## Blocked claims

- `l4e-inhibition-window` and `l4e-whisking-inhibition` (Fig. 3): data missing. The six
  neurons recorded at rest and depolarized near 0 mV by current injection are not in the data
  set. No trial of a whole-cell L4 E neuron, JY0978 (the cell list's Fig. 3 example) included,
  has a median Vm above -42.8 mV, and their largest stored current is 0.2 nA. Every ephys trial
  is read (the counts match the whisker trials). The only trials near 0 mV are the first 18 of
  the FS neuron JY1139, which keeps firing in them with no current above 0.1 nA: a drifting
  recording, not the protocol of Fig. 3, which inactivates spikes with about 1 nA. The FS neuron
  JY1171 carries a steady current above 0.5 nA in every trial at a median Vm of -40.7 mV, also
  not that protocol. No part of either claim can be tested (printed below).
- `thalamus-inhibition-vpm`: data missing (above).
- `rabies-vpm-input`: data missing. The data set holds no histology.
- `ion-cut-l5-fs`: data missing. The paper shows no data ("data not shown"), and the ION-cut
  folders hold only L4 neurons (depths are not stored in the cell-attached files; the cell list
  gives 444 to 614 um for the ION-cut FS neurons).

In [ ]:
dep = {}
for c in cells:
    if c["vm"] is None:
        continue
    med = c["ephys_trials"]["median_vm"]
    cur = c["ephys_trials"]["max_abs_current"]
    dep[ssc7.cell_name(c)] = (GROUP[ssc7.cell_name(c)], float(np.nanmax(med)), float(np.nanmax(cur)))
D = pd.DataFrame(dep, index=["group", "max_trial_median_vm", "max_abs_current_nA"]).T
print("largest per-trial median Vm by group (mV), over every ephys trial:", D.groupby("group").max_trial_median_vm.max().round(1).to_dict())
print("largest stored current (nA) by group:", D.groupby("group").max_abs_current_nA.max().round(3).to_dict())
print("JY0978 (the cell list's Fig. 3 example):", WCT[WCT.name.str.startswith("JY0978")].to_string(index=False))
# The one neuron with trials near 0 mV: JY1139 (FS folder; cell list FS sheet, PV-ChR2 mouse)
c = NAME["JY1139AAAA"]
med = c["ephys_trials"]["median_vm"]
spk = ssc7.spikes_by_trial(c)
hi = np.flatnonzero(med > -30)
print(f"JY1139: trials with median Vm above -30 mV: {hi.tolist()} (median Vm {med[hi].round(1).tolist()}); spikes in those trials {[len(spk[i]) for i in hi]}; "
      f"largest |current| in them {np.nanmax(c['ephys_trials']['max_abs_current'][hi]):.4f} nA; median Vm of trials 20 on {np.median(med[20:]):.1f} mV")
# neurons with currents above 0.5 nA: how much of each trial
for n in WCT[WCT.max_abs_current_nA > 0.5].name:
    cc = NAME[n]
    big = np.nansum(np.abs(cc["current"]) > 0.5, 1)
    print(f"{n} ({GROUP[n]}): trials with any 1 ms bin above 0.5 nA {int((big > 0).sum())} of {len(big)}; most bins above 0.5 nA in one trial {int(big.max())} ms; "
          f"median Vm of those trials {np.median(cc['ephys_trials']['median_vm'][big > 0]):.1f} mV")
dep_ion_fs = pd.to_numeric(CL[CL.section == "FS neurons, ION cut"].depth, errors="coerce")
print(f"cell list depths of the ION-cut FS neurons: {dep_ion_fs.min():.0f} to {dep_ion_fs.max():.0f} um (n = {dep_ion_fs.notna().sum()})")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Of 48 claims, 43 were tested and 5 blocked. Of those tested, 24 reproduced, 13 partially
reproduced and 6 did not reproduce.

| Claim | Outcome |
|---|---|
| behavior-performance | partially reproduced (assumed) |
| vpm-baseline-rate | partially reproduced |
| vpm-touch-response | partially reproduced |
| vpm-whisking-count | reproduced |
| vpm-whisking-rate | reproduced |
| vpm-phase-modulation | reproduced |
| vpm-rate-amplitude-trials | reproduced |
| l4e-example-e1 | reproduced |
| l4e-example-e24-e32 | reproduced |
| l4e-resting-vm | reproduced |
| l4e-touch-psp | partially reproduced |
| l4e-whisking-vm | reproduced |
| l4e-distance-threshold | partially reproduced |
| l4e-reversal-potential | reproduced |
| l4e-hf-fluctuations | reproduced |
| l4e-inhibition-window | blocked: data missing |
| l4e-whisking-inhibition | blocked: data missing |
| fs-baseline-rate | reproduced |
| fs-touch-vm | reproduced |
| fs-touch-spikes | partially reproduced (assumed) |
| fs-whisking-count | reproduced |
| fs-whisking-vm | partially reproduced |
| fs-whisking-rate | reproduced |
| fs-phase-coupling | partially reproduced |
| fs-hf-fluctuations | reproduced |
| l4e-spikes-touch-selective | not reproduced |
| thalamus-inhibition-vpm | blocked: data missing |
| thalamus-inhibition-l4e-vm | partially reproduced (assumed) |
| thalamus-inhibition-fs | partially reproduced |
| thalamus-inhibition-dose | reproduced |
| rabies-vpm-input | blocked: data missing |
| ion-cut-behavior | reproduced |
| ion-cut-l4e-touch | not reproduced |
| ion-cut-l4e-whisking | partially reproduced |
| ion-cut-l4e-phase | reproduced |
| ion-cut-l4e-spectra | reproduced |
| ion-cut-fs-touch | reproduced |
| ion-cut-fs-whisking | reproduced |
| ion-cut-fs-phase | reproduced |
| ion-cut-l5-fs | blocked: data missing |
| enphr-kruskal-wallis | reproduced |
| enphr-l4e-rates | partially reproduced |
| enphr-whisking-driven | reproduced |
| enphr-spikes-per-touch | not reproduced |
| enphr-auc | not reproduced (assumed) |
| enphr-auc-groups | not reproduced |
| enphr-pv-latency | partially reproduced |
| enphr-behavior | not reproduced |

Three tests fixed by the claims file cannot tell the effect from its null well, and are asked
of a reader: ion-cut-l4e-touch (a peak measure tested against zero is positive with no
response), thalamus-inhibition-dose (with control as zero power, the correlation mostly tests
light against no light) and enphr-auc (the evoked rate and the control AUC share their trials;
here the split-half check finds no bias).

### How the analysis differs from the paper's

- The principal whisker of cortical files with two tracked whiskers is taken to be the first
  series, as the data set's example code takes it. The files do not say.
- Light trials are told apart by the photostimulus series that is on in each trial (its site and
  wavelength), mapped to the three experiments of the Online Methods. Light effects are measured
  inside each trial's light period (M-LIGHT reading a), with whole trials as reading (b).
- The light-onset latency rule of the claims file finds a crossing at the onset sample itself,
  where the light produces a step in the average Vm. A variant with 1 ms smoothing and a 1 ms
  hold is used as an assumption.
- The touch-speed cutoff of the example code (`prct_speed`) is not applied. Touches are not split
  by direction.
- Spike thresholds come from the data set's spike times, searched back to the dVm/dt crossing.
  For neurons with few spikes the threshold line extrapolates far; a line held at its end values
  and a per-neuron threshold are shown as variants.
- Whole-cell trials whose median Vm lies more than 20 mV from the neuron's median are left out of
  the Vm measures (13 trials of JY1139, an FS recording that drifts from near 0 mV), an
  assumption after the Methods' rejection of drifting recordings.
- In whisking-onset windows, Vm after a later touch is set to NaN (M-EPOCH). Two outcomes sit at
  their margins because of it: l4e-reversal-potential (4.9 against 5 mV) and
  ion-cut-l4e-whisking (control mean 1.337 against a bound of 1.34).
- The first touch of trials tagged passive is left out everywhere, the ROC of enphr-auc included;
  enphr-auc reproduces only with those touches kept.
- Where the claims file lists readings, every reading is shown, and the outcome is the one the
  criterion gives under them.

### What was found about the data set

- The whisker timestamps of the cortical files mix units: the trial start is in milliseconds and
  the time within a trial in seconds.
- Which tracked whisker series is the principal whisker is not recorded. The order of
  `whisker_configuration` does not give it.
- The thalamic multiunit recordings under TRN-axon photostimulation, the rabies tracing, the L5 FS
  neurons after ION cut and the recordings with depolarizing current injection are not in the
  data set. The behavior sessions of Supplementary Fig. 14 may not be either: the recording
  sessions give 9 mice where the paper has 8.
- Eight whole-cell L4 E files flag 132 stimulus trials but hold no photostimulus series, and two
  more have 23 flagged trials without light in their series.
- The neuron JY1139 in the whole-cell FS folder is described in its file as "Excitatory", and
  its first 18 trials sit near 0 mV while it fires, with no current injected.
- The cell list does not name JY1268, in the cell-attached L4 E folder, and it counts JY0781 among
  the PV-IRES-Cre L4 E neurons, though its file holds no eNpHR light.
- The eNpHR experiments hold 43 L4 E neurons with constant-light trials, against the paper's 40.